# Build RBA stoich for GAMS (originally A01_build_excel_stoich_for_GAMS.ipynb)

#### Load data

In [1]:
import cobra,json,os,sys,re,pandas as pd,ast,yaml
from collections import OrderedDict
from copy import deepcopy
# if running from another notebook, retain the 'decompartmentalize' variable if it exists; otherwise, set to False (so that decompartmentalization is only performed if explicitly specified in the notebook)
decompartmentalize = sys.modules['__main__'].__dict__.get('decompartmentalize', False)
load_model = False if decompartmentalize else True

model_root_path = '../'
sys.path.append(model_root_path)
from pycore import *
SM_path = SM_path if not decompartmentalize else sys.modules['__main__'].__dict__.get('SM_path_decomp', None)

warnings_summary = [] # msgs to print later
def warn(msg,ignore_duplicate_warnings=True):
	"""Append a warning message to the warnings list (raises warnings at the end of the script) and print it (for easier debugging)."""
	if ignore_duplicate_warnings and msg in warnings_summary:
		return
	warnings_summary.append(msg)
	print("Warning:", msg)

/Users/ejm6426/Documents/pkRBA/pycore/../build_model/input/iIsor850_RBA.json


In [2]:
# If you have to use multiple stoichiometric models (SMs), 
#   it's best to keep 1 as the main copy and then use transaction files to apply differences to the others. Helps improve consistency in annotations, stoich. etc.

# List parameters for your model version(s), such as:
	# max_allowed_mito_proteome_allo_fraction: maximum fraction of proteome that can be allocated to mitochondria

# store model.compartments in a file
with open('./model/compartments.txt', 'w') as f:
	f.write('\n'.join(['/'] + [f"'{i}'" for i in model.compartments] + ['/']))
with open('./model/compartments_mito.txt', 'w') as f:
	f.write('\n'.join(['/'] + [f"'{i}'" for i in RBA_defaults.mito_compartments] + ['/']))

# Update GAMS settings file to reflect model version, 
#   and add NGAM, mito capacity, etc.
#   If not using multiple models, replace everything in the curly braces with the values you want
with open('./model/model-version-settings.txt', 'w') as f:
	mampaf = getattr(RBA_defaults, 'max_allowed_mito_proteome_allo_fraction', 1)
	f.write(f"* file with settings updated by build_model scripts, to help with managing multiple SMs\n"
		f"$setGlobal max_allowed_mito_proteome_allo_fraction {mampaf}\n"
		)

# Protein
df_pro = read_spreadsheet('./input/PROTEIN_stoich_curation.tsv')
# set 1st column as index
df_pro.index = df_pro[df_pro.columns[0]]
## OPTIONAL: add "special cases only" sheet to df_pro. 
##   Good for distinguishing proteins in the organism from ones added for testing purposes.
include_special_case_proteins = False # affects protein and enzyme stoichiometry
if include_special_case_proteins:
	df_pro_special = read_spreadsheet('./input/PROTEIN_stoich_curation.tsv',sheet_name='special cases only',header=1)
	df_pro = pd.concat([df_pro,df_pro_special],ignore_index=True)


# Amino acid info
df_aamap = pd.read_csv('./input/PROTEIN_amino_acid_map.txt', sep='\t')
df_aamap.index = df_aamap.aa_abbv.to_list()

# Dummy protein (optional)
dummy_path = './input/PROTEIN_dummy_prot_calc.xlsx'
dummy_list = []
if os.path.isfile(dummy_path):
	# 1st entry is default dummy protein
	if getattr(RBA_defaults, 'dummy_sheet_name', None):
		dummy_list.append({"df":read_spreadsheet(dummy_path,sheet_name=RBA_defaults.dummy_sheet_name),"name":"PROSYN-PROTDUMMY"})
	else:
		dummy_list.append({"df":read_spreadsheet(dummy_path),"name":"PROSYN-PROTDUMMY"})
	# remove unless needed
	dummy_list.append({"df":read_spreadsheet(dummy_path,sheet_name='unidentified'),"name":"PROSYN-PROTDUMMYUNIDENTIFIED"})
	for prot in dummy_list:
		prot["df"].index = prot["df"].aa_abbv.to_list()
		prot["medianL"] = int(round(prot["df"].at['A', 'Unnamed: 5'], 0))
		# Add 1e-5 g/mmol to prevent number round lost of protein MW
		#dummy_MW
		prot["MW"] = round(prot["df"].at['C', 'Unnamed: 5'], 5) + 1e-5
		prot["biosyn"] = prot["name"].replace('PROSYN-','BIOSYN-',1)
		prot["met"] = prot["name"].lower().replace('prosyn-','BIO-',1)
		prot["translation_loc"] = 'unknown' if not 'translation_loc' in prot else prot['translation_loc']

# Enzyme
df_enz_path = './input/ENZYME_stoich_curation.tsv'
if os.path.isfile(df_enz_path):
	df_enz = read_spreadsheet(df_enz_path)
	# add 'id' column to df_enz, which is 'RXN-' + rxn_src + '_' + dir + '-' + enz
	if include_special_case_proteins:
		df_enz_special = read_spreadsheet('./input/ENZYME_stoich_curation_special_cases.tsv')
		df_enz = pd.concat([df_enz,df_enz_special],ignore_index=True)

# RNA (optional)
RNA_path = './input/RNA_stoich.xlsx'
df_rnas = pd.DataFrame(columns=['RNAid','A'])
if os.path.isfile(RNA_path):
	df_rnas = read_spreadsheet('./input/RNA_stoich.xlsx')
df_rnas.index = df_rnas.RNAid.to_list()

# Ribosome (optional)
loaded_data = {}
ribo_path = './input/ribosome_composition.yaml'
if os.path.isfile(ribo_path):
	with open(ribo_path, 'r') as f:
		loaded_data = yaml.load(f, Loader=yaml.FullLoader)

ribo_dict = {}
for k, v in getattr(RBA_defaults, 'ribo_dict', {}).items():
	ribo_dict[k] = pd.DataFrame()
	if os.path.isfile(v):
		ribo_dict[k] = {'k_ribo': RBA_defaults.k_ribo_max, 'subunits': read_spreadsheet(v)}
	else:
		# construct ribo_dict from loaded_data if file doesn't exist
		ribo_dict[k] = loaded_data.get(k[1:], {})
		# convert the "subunits" dict back to a DataFrame format for consistency with original ribo_dict structure; split keys by " OR " to get paralogs, and add 'n' column for stoichiometry if needed
		subunits_data = []
		for subunit_str, n in ribo_dict[k].get('subunits', {}).items():
			proteins = subunit_str.split(' OR ')
			for i2, protein in enumerate(proteins):
				subunits_data.append({'id': protein, 'n': n})
				if i2 > 0:
					# add '1' to 'paralog' column for all but the first protein to indicate they are paralogs
					subunits_data[-1]['paralog'] = 1
		ribo_dict[k]['subunits'] = pd.DataFrame(subunits_data)

# Biomass
# heading start at cell X4
# if 'biomass_sheet_id' is not specified in RBA_defaults, use the 1st sheet in the file
biomass_path = './input/BIOMASS_RBA.xlsx'
if os.path.isfile(biomass_path):
	if getattr(RBA_defaults, 'biomass_sheet_name', None):
		df_biom = read_spreadsheet(biomass_path, sheet_name=RBA_defaults.biomass_sheet_name, header=4)
	else:
		print('No biomass sheet name specified in RBA_defaults, using 1st sheet in file')
		df_biom = read_spreadsheet(biomass_path, sheet_name=0, header=4)
else:
	# read biomass_composition.yaml if biomass_path doesn't exist
	biomass_yaml_path = './input/biomass_composition.yaml'
	if os.path.isfile(biomass_yaml_path):
		with open(biomass_yaml_path, 'r') as f:
			biomass_data = yaml.load(f, Loader=yaml.FullLoader)
		# convert biomass_data to a DataFrame format similar to df_biom, with 'id' key used in 'id' column, and 'metabolites' keys formatted into 'reaction' column
		df_biom = pd.DataFrame(columns=['rxn_id', 'rxn_equation'])
		# add row for each biomass reaction in biomass_data
		for biomass_rxn_id, biomass_rxn_data in biomass_data.items():
			df_biom.loc[len(df_biom)] = [biomass_rxn_id, build_reaction_equation_from_metabolites_dict_RBA(biomass_rxn_data, arrow='-->')]
# list of biomass rxns to turn off by default and only turn on when needed; any rxns not found will be excluded from list
whole_biomass_rxns = ['BIOSYN-' + i for i in ['BIODILAERO', 'BIODILAERO-NOGAM','BIOXYL-PINHEIRO','BIOXYL-PINHEIRO-NOGAM']] + ['BIOMASS_xyl_hybrid']

# checks which rxns to ignore when seeing if all necessary enzymes are modeled. Ignore blocked rxns (to avoid unnecessary work from adding them)
rxns_not_needing_enzymes = list(dict.fromkeys([r.id for r in model.reactions if r.bounds == (0,0) and r.id not in RBA_defaults.must_include_rxns] + [rxn.id for rxn in find_biomass_reactions(model)] + ['BIOMASS_MFA','BIOMASS_MFA_NO_GAM','BIOMASS_RBA']))

# take IDs of all rxns
print('Reactions to ignore when checking if enzymes must be added:')
print(rxns_not_needing_enzymes)

Reactions to ignore when checking if enzymes must be added:
['PUNP1_c', 'PUNP1_m', 'PUNP7_c', 'ACONTa_c', 'ASPt2_m', 'TYRtps_v', 'GNNUC_c', 'ILEtps_v', 'HCO3t_c_n', 'CHLSTI_c', 'LEUtps_v', 'GLNtps_v', 'ASPtps_v', 'ASNtps_v', 'GLUtps_v', 'ADNUC_c', 'BIOMASS_MFA_NO_GAM', 'Biomass_MFA_succ_macros_only', 'Biomass_MFA_succ_raw', 'Biomass_MFA_succ_raw_normalized', 'Biomass_MFA_succ_with_MFA_extras', 'Biomass_MFA_succ_with_MFA_extras_with_GAM', 'Biomass_Isor', 'Biomass_MFA', 'BIOMASS_MFA', 'BIOMASS_RBA']


In [3]:
df_eqn = pd.DataFrame(columns=['id', 'type', 'coupling_type', 'coupling_species', 'reaction','tag','FBA_name','dir','enz_id','compartment'])

In [4]:
# Expanding df_pro to include sublocations as separate proteins
mw = 'MW (g/mmol)'
# calculate MW using df_aamap and the amino acid composition in df_pro
if mw not in df_pro.columns:
	for i in df_pro.index:
		prot = df_pro.at[i]
		aa_composition = prot['sequence']
		prot_mw = 0
		for aa in aa_composition:
			if aa in df_aamap.index:
				prot_mw += df_aamap.at[aa, 'MW']
			else:
				warn(f"amino acid '{aa}' not found in amino acid map. Skipping.")
		df_pro.at[i, mw] = prot_mw
# remove the 'id' column
if 'id' in df_pro.columns:
	df_pro = df_pro.drop(columns=['id'])
# if the 'gene_src' column is empty, fill it with the 'name' or 'uniprot' column
for col in ('name', 'uniprot'):
	if col in df_pro.columns:
		df_pro['gene_src'] = df_pro['gene_src'].fillna(df_pro[col])
# if there are any entries with no gene_src, raise an error
if df_pro['gene_src'].isnull().any():
	missing_genes = df_pro[df_pro['gene_src'].isnull()]
	raise ValueError(f"The following proteins are missing 'gene_src' information:\n{missing_genes}")
# for each protein, if there are commas in subloc_assigned, split them into separate entries with 'id' modified to reflect sublocation
df_pro['subloc_assigned'] = df_pro['subloc_assigned'].apply(lambda x: x.split(',') if type(x) == str and ',' in x else x)
df_pro = df_pro.explode('subloc_assigned').reset_index(drop=True)

In [5]:
# translating PTM_stoich_edits into final versions read by model.

# for all entries where PTM_stoich_edits is a dict, assign it values from the 'default' key if it exists, otherwise assign it value from the key for its sublocation
def assign_ptm_stoich(row):
	# check if PTM_stoich_edits can be converted to a dict, if so, convert it
	if type(row['PTM_stoich_edits']) == str:
		try:
			row['PTM_stoich_edits'] = ast.literal_eval(row['PTM_stoich_edits'])
		except (ValueError, SyntaxError):
			pass
	if type(row['PTM_stoich_edits']) == dict:
		ptm_stoich_dict = row['PTM_stoich_edits']
		if row['subloc_assigned'] in ptm_stoich_dict:
			row['PTM_stoich_edits'] = ptm_stoich_dict[row['subloc_assigned']]
		elif 'default' in ptm_stoich_dict:
			row['PTM_stoich_edits'] = ptm_stoich_dict['default']
	return row
df_pro = df_pro.apply(lambda row: assign_ptm_stoich(row), axis=1)

# if there are pipes in PTM_stoich_edits, split them into separate entries
df_pro['PTM_stoich_edits'] = df_pro['PTM_stoich_edits'].apply(lambda x: x.split('|') if type(x) == str and '|' in x else x)
df_pro = df_pro.explode('PTM_stoich_edits')

In [6]:
# add the 'id' column back in, assigning it to gene_src plus subloc_assigned if the latter is not 'unknown'
# modify IDs to reflect sublocation
# add location only if there are multiple proteins with the same gene source
if 'id' not in df_pro.columns:
	has_entry = 'Entry' in df_pro.columns
	has_protein_col = 'protein' in df_pro.columns
	df_pro['id_without_compartment'] = df_pro.apply(
		lambda row: (
			str(row['gene_src']) if pd.notna(row['gene_src'])
			else str(row['Entry']) if has_entry and pd.notna(row['Entry'])
			else str(row['protein']) if has_protein_col and pd.notna(row['protein'])
			else str(row.name)
		),
		axis=1
	)
	# add sublocation only if multiple proteins have the same ID (for continuity w/ past RBA versions; unclear if needed, though having consistent IDs would help futureproof other scripts)
	df_pro['id'] = df_pro.apply(
		lambda row: (
			re.sub(r'^[^-]*-','',str(met_ID(tag=ptmpro_prefix, base_id=str(row['id_without_compartment']),
		  	PTMs=row['PTM_stoich_edits'] if pd.notna(row['PTM_stoich_edits']) and row['PTM_stoich_edits'] != '' else {},
			compartment=row['subloc_assigned'] if pd.notna(row['subloc_assigned']) and row['subloc_assigned'] != 'unknown' and df_pro[df_pro['id_without_compartment'] == row['id_without_compartment']].shape[0] > 1 else None,
			)))
		),
		axis=1
	)
	for i in df_pro.index:
		if len(df_pro.at[i, 'id']) > max_id_length_for_ptmpro:
			# if ID is too long, and there's only 1 row with this id_without_compartment that has PTMs, then remove PTM info from the ID
			ptm_mask = (
				(df_pro['id_without_compartment'] == df_pro.at[i, 'id_without_compartment']) &
				(df_pro['PTM_stoich_edits'].notna()) &
				(df_pro['PTM_stoich_edits'] != '')
			)
			if ptm_mask.sum() == 1:
				new_id = vars(met_ID.from_name(f'{ptmpro_prefix}-{df_pro.at[i, "id"]}'))
				new_id.pop('PTMs', None)
				new_id = str(met_ID(**new_id)).removeprefix(f'{ptmpro_prefix}-')
				if new_id not in df_pro['id'].values:
					print(f"Warning: protein ID '{df_pro.at[i, 'id']}' is too long but there is only 1 entry with this base ID and PTMs, so PTM info has been removed from the ID to shorten it. Consider shortening the base ID or removing PTM info from the spreadsheet for this protein to avoid this in the future.")
					df_pro.at[i, 'id'] = new_id
			else:
				print(f"Warning: protein ID '{df_pro.at[i, 'id']}' is too long but there are multiple entries with the same base ID and PTMs, so PTM info cannot be removed from the ID. Consider shortening the base ID or removing PTM info from the spreadsheet for this protein.")
	if 'name' not in df_pro.columns:
		df_pro['name'] = df_pro['id_without_compartment']
df_pro.index = df_pro.id.to_list()
# save df_pro to model folder
df_pro[['id'] + [col for col in df_pro.columns if col != 'id']].to_csv('./model/PROTEIN_stoich_curation.tsv', sep='\t', index=False)

In [7]:
# if there are any duplicates in df_pro index, raise error
if df_pro.index.duplicated().any():
	dup_ids = df_pro.index[df_pro.index.duplicated()].unique()
	raise ValueError(f"Duplicate protein IDs found in PROTEIN_stoich_curation.tsv after processing sublocations: {', '.join(dup_ids)}")

In [8]:
# list all genes, to map them to translation (PROSYN) rxns and other rxns making their products where needed
# combine all genes from df_pro and model
# combine all genes from df_pro and model, ensuring all are strings and not NaN
genes_model = [str(gene.id) for gene in model.genes if pd.notna(gene.id)]
genes_pro = [str(g) for g in df_pro.gene_src.to_list() if pd.notna(g)]
genes_rna = [str(r) for r in df_rnas.RNAid.to_list() if pd.notna(r)]
genes = sorted(list(dict.fromkeys(genes_model + genes_pro + genes_rna)))
gene_expression_dict = {}

#### Assemble reactions

In [9]:
# Automatically create placeholders for enzymes based on GPRs in model.reactions
rxns_to_skip = set(rxns_not_needing_enzymes)
enz_rxn_dict = {}
spont_rxn_dict = {}
rxn_updates = {rxn.id: {} for rxn in model.reactions}
GPR_enz_dict = {}
try:
	df_enz_rxns = read_spreadsheet('./input/ENZYME_stoich.tsv')
	# make set of rxns to skip, from combining the 'rxns' column cells in df_enz_rxns (to avoid reviewing existing rxns) and separating by commas. However, relying on this will prevent updates to their GPRs from being added to the model.
	# rxns_to_skip.update(set(df_enz_rxns.rxns.str.split(',').sum()))
	# make enz_rxn_dict from df_enz_rxns, with 'enz' as keys and 'rxns' and 'protein_stoich' as values
	for index, row in df_enz_rxns.iterrows():
		enz_rxn_dict[row['enz']] = {'rxns': sorted(list(dict.fromkeys(row['rxns'].split(',')))),
										'protein_stoich': {prot.split(':')[0]: float(prot.split(':')[1]) for prot in row['protein_stoich'].split(',')}}
		if not pd.isna(row[mw]):
			enz_rxn_dict[row['enz']][mw] = row[mw]
except:
	df_enz_rxns = pd.DataFrame(columns=['enz','rxns','protein_stoich',mw])
# add dict of protein_stoich to enz_rxn_dict, with each protein as a key and its stoichiometry as the value

# find all unique GPRs in model.reactions, and their associated reactions
for rxn in model.reactions:
	# print(rxn.id, rxn.gene_reaction_rule)
	if rxn.id in rxns_to_skip:
		continue
	if rxn.gene_reaction_rule not in RBA_defaults.spont_GPRs + ['']:
		if rxn.gene_reaction_rule in GPR_enz_dict.keys():
			# add rxn to corresponding enz in enz_rxn_dict
			enz_ids = GPR_enz_dict[rxn.gene_reaction_rule]
			for enz_id in enz_ids:
				enz_rxn_dict[enz_id]['rxns'].append(rxn.id)
				rxn_updates[rxn.id].setdefault('enz',enz_id)
		else:
			GPR_enz_dict[rxn.gene_reaction_rule] = set()
			enzs = sorted(find_true_combinations(rxn.gene_reaction_rule, use_knockout=True, variables=[g.id for g in rxn.genes]), key=lambda x: sorted(x.keys()))
			for enz in enzs:
				# print('\t',enz)
				enz_found = None
				# default: 1 copy of each protein is used
				# prot_stoich = ','.join([gene+':1' for gene in sorted(enz.keys()) if enz[gene]])
				prot_stoich = {gene:1 for gene in sorted(enz.keys()) if enz[gene] and gene not in RBA_defaults.spont_GPRs}
				# check if any enzymes with the same stoichiometry are already in enz_rxn_dict
				enz_id = ''.join(sorted(prot_stoich.keys()))

				for old_enz_id in enz_rxn_dict:
					# print(enz_id,enz_rxn_dict[enz_id])
					# automatically assumes enzymes are the same if they have the same proteins, regardless of order/count
					if sorted(enz_rxn_dict[old_enz_id]['protein_stoich'].keys()) == sorted(prot_stoich.keys()):
						enz_rxn_dict[old_enz_id]['rxns'].append(rxn.id)
						enz_found = old_enz_id
						break
				if not enz_found:
					# enz_rxn_dict[rxn.id] = enz
					enz_rxn_dict[enz_id] = {'rxns':[rxn.id],
											'protein_stoich':prot_stoich}
					rxn_updates[rxn.id].setdefault('enz',enz_id)
					GPR_enz_dict[rxn.gene_reaction_rule].add(enz_id)
				else:
					GPR_enz_dict[rxn.gene_reaction_rule].add(enz_found)
	else:
		# add to spont_rxn_dict under its GPR as a list of rxn IDs, if it is not already there
		spont_rxn_dict[rxn.gene_reaction_rule] = spont_rxn_dict.get(rxn.gene_reaction_rule, {'rxns':[],'protein_stoich':{'zeroCost':1}})
		spont_rxn_dict[rxn.gene_reaction_rule]['rxns'].append(rxn.id)
		rxn_updates[rxn.id].setdefault('gene_reaction_rule',rxn.gene_reaction_rule)
		# spont_rxn_dict[rxn.gene_reaction_rule] = spont_rxn_dict.get(rxn.gene_reaction_rule, [rxn.id])
		# spont_rxn_dict[rxn.gene_reaction_rule] = rxn.id
					
# give each enz in enz_rxn_dict a row in df_enz_rxns
new_rows = []
for enz in enz_rxn_dict:
	# only add enzymes that are not already in df_enz_rxns
	if df_enz_rxns[df_enz_rxns.enz == enz].empty:
		# calculate MW of each enzyme based on its protein stoichiometry
		if mw in enz_rxn_dict[enz] and enz_rxn_dict[enz][mw]:
			enz_mw = enz_rxn_dict[enz][mw]
		else:
			enz_mw = 0
			for prot in enz_rxn_dict[enz]['protein_stoich']:
				pro_mw = df_pro.at[prot,mw] if prot in df_pro.index else 0
				# check in gene_src column
				if pro_mw == 0:
					if prot in df_pro.gene_src.to_list():
						# print(prot,df_pro.at[df_pro.gene_src == prot,mw])
						# if there are multiple proteins with the same gene_src, check if their MW is the same
						# if not, print a warning
						# pick the first one
						pro_match = df_pro.loc[df_pro.gene_src == prot,mw].to_list()
						if df_pro.loc[df_pro.gene_src == prot,mw].nunique() > 1:
							warn(f'{prot} has multiple MWs in df_pro')
						pro_mw = pro_match[0]
					else:
						warn(f'{prot} not found in df_pro')
				enz_mw += pro_mw*enz_rxn_dict[enz]['protein_stoich'][prot]
		new_rows.append({'enz':enz,'rxns':','.join(enz_rxn_dict[enz]['rxns']),
						'protein_stoich':','.join([prot+':'+str(enz_rxn_dict[enz]['protein_stoich'][prot]) for prot in enz_rxn_dict[enz]['protein_stoich']]),
						mw:enz_mw,
						'status':'automatically added'})
df_enz_rxns = pd.concat([df_enz_rxns, pd.DataFrame(new_rows)], sort=False, ignore_index=True)
# sort by id, MW, and then by rxns
df_enz_rxns = df_enz_rxns.sort_values(by=['rxns',mw,'enz'])
# save automatically updated df_enz_rxns as separate file
df_enz_rxns.to_csv('./model/ENZYME_stoich.tsv', sep='\t', index=False)

In [10]:
# check if all rxns in model are accounted for in df_enz_rxns, and if they have been automatically updated
import itertools
# make a list of all rxns in df_enz_rxns
enz_rxns = []
for index, row in df_enz_rxns.iterrows():
	enz_rxns += row['rxns'].split(',')
enz_rxns = sorted(list(dict.fromkeys(enz_rxns)))
# check if all rxns in model are in enz_rxns
missing_rxns = []
for rxn in model.reactions:
	if rxn.id not in rxns_not_needing_enzymes and rxn.id not in enz_rxns and rxn.id not in whole_biomass_rxns and rxn.gene_reaction_rule not in spont_rxn_dict.keys():
		missing_rxns.append(rxn.id)
if len(missing_rxns) > 0:
	print(f'Warning: the following reactions are not accounted for in df_enz_rxns:')
	for rxn in missing_rxns:
		print(rxn)
# remove rxns from rxn_updates if their value is {}
for rxn in rxn_updates.copy():
	if not rxn_updates[rxn]:
		del rxn_updates[rxn]
	else:
		print(f'Updating {rxn} with {rxn_updates[rxn]}')
# if rxn_updates isn't empty, print a warning
# if len(rxn_updates) > 0:
# 	raise ValueError(f'Warning: the following reactions have been automatically updated:\n{rxn_updates}')
# convert df_enz_rxns to old ENZYME_stoich_curation file format from 2024 version of scRBA, for compatibility
df_rxn_enz_pairs = pd.DataFrame(columns=['id','rxn_src','enz','protein_stoich'])
catalyst_rxn_dict = {**spont_rxn_dict, **enz_rxn_dict}
for k, v in catalyst_rxn_dict.items():
	if k in spont_rxn_dict:
		catalyst_rxn_dict[k]['rxns'].extend(spont_rxn_dict[k]['rxns']) # adds rxns that were overrided by merger with enz_rxn_dict
		catalyst_rxn_dict[k]['rxns'] = list(dict.fromkeys(catalyst_rxn_dict[k]['rxns']))
for enz in catalyst_rxn_dict:
	enz_id = enz if enz != '' else 'SPONT'
	enz_stoich_combinations = ['zeroCost'] if enz in RBA_defaults.spont_GPRs + [''] else []
	# don't seem to need MW of enzyme in prior RBA code
	for rxn in catalyst_rxn_dict[enz]['rxns']:
		dirs = []; model_rxn = model.reactions.get_by_id(rxn)
		update_enz_id = False
		compartments_to_add = set()
		if enz_stoich_combinations != ['zeroCost']:
			enz_stoich_options = []
			rxn_compartments = model_rxn.compartments
			for c,prot in enumerate(catalyst_rxn_dict[enz]['protein_stoich']):
				enz_stoich_options.append([set(),None]) # list of [set of protein IDs, stoich value] for each prot in the enzyme, to be used for generating enzyme versions if needed. Using set to avoid duplicates when multiple entries in df_pro match the same gene source.
				prot_id = None
				prot_matches = pd.DataFrame()
				# set prot_matches to all entries in df_pro with index matching prot and subloc_assigned matching one of the rxn compartments
				prot_matches = df_pro[(df_pro.index == prot) & (df_pro['subloc_assigned'].apply(lambda x: x in rxn_compartments if pd.notna(x) else False))]
				if prot_matches.empty:
					update_enz_id = True
					# check gene_src column
					prot_matches = df_pro[df_pro.gene_src == prot]
					if not prot_matches.empty:
						# check compartment-protein maps dict in df_enz_rxns
						enz_idx = df_enz_rxns[df_enz_rxns['enz'] == enz].index[0]
						# if multiple matches, check if any have subloc_assigned in rxn_compartments
						if prot_matches.shape[0] > 1:
							for idx, match in prot_matches.iterrows():
								if not ((match['subloc_assigned'][0] if type(match['subloc_assigned']) == pd.Series else match['subloc_assigned']) in rxn_compartments):
									prot_matches = prot_matches.drop(idx)
						if pd.notna(df_enz_rxns.at[enz_idx, 'compartment-protein_maps']):
							cp_map = ast.literal_eval(df_enz_rxns.at[enz_idx, 'compartment-protein_maps'])
							# parse cp_map, checking the rxn's compartment for the prot value and using its associated prot_id
							for comp,v in cp_map.items():
								if comp in rxn_compartments and prot in v:
									prot_id = cp_map[comp][prot]
									# include in prot_matches
									prot_matches = pd.concat([prot_matches, df_pro[(df_pro.gene_src == prot) & (df_pro['subloc_assigned'] == cp_map[comp][prot].split('_')[-1])]], ignore_index=True)
									print(f'prot_id set to {prot_id} for {prot} via compartment-protein map')
									break
				for idx, match in prot_matches.iterrows():
					enz_stoich_options[c][0].add(match['id'])
					enz_stoich_options[c][1] = catalyst_rxn_dict[enz]['protein_stoich'][prot]
					compartments_to_add.add(match['subloc_assigned'][0] if type(match['subloc_assigned']) == pd.Series else match['subloc_assigned'])
				if prot_matches.empty:
					warn(f'protein {prot} not found in df_pro for reaction {rxn} with compartments {rxn_compartments}.')
			# print(f'enz_stoich_options for {enz} and {rxn}: {enz_stoich_options}')
			combos = list(itertools.product(*[itertools.combinations_with_replacement(subunit[0], int(subunit[1])) for subunit in enz_stoich_options if subunit[0] and subunit[1]]))
			enz_stoich_combinations = [','.join([','.join([f'{prot}:{counts.count(prot)}' for prot in sorted(set(counts))]) for counts in combo]) for combo in combos]
			# print(f'enz_stoich_combinations for {enz} and {rxn}: {enz_stoich_combinations}')
			# # convert combos to list of dicts with prot as key and stoich as value
			# enz_stoich_combinations = [[{prot: counts.count(prot) for prot in set(counts)} for counts in combo] for combo in combos]
		
		enz_id_final = enz_id
		if update_enz_id:
			# update enz_id to include sublocations
			enz_id_final = '_'.join([enz_id] + sorted(list(compartments_to_add)))
		# add FWD and REV to rxn_src if those directions are possible
		if model_rxn.lower_bound < 0 or (model_rxn.bounds == (0,0) and rxn in RBA_defaults.must_include_rxns):
			dirs.append('REV')
		if model_rxn.upper_bound > 0 or (model_rxn.bounds == (0,0) and rxn in RBA_defaults.must_include_rxns):
			dirs.append('FWD')
		for prot_stoich_str in list(dict.fromkeys(enz_stoich_combinations)):
			for dir in dirs:
				# print(enz,rxn)
				# df_rxn_enz_pairs = df_rxn_enz_pairs.append({'id':f'RXN-{rxn}_{dir}-{enz}','enz':enz,'gpr':'','protein_stoich':','.join([prot+':'+str(enz_rxn_dict[enz]['protein_stoich'][prot]) for prot in enz_rxn_dict[enz]['protein_stoich']]),'status':'automatically added'}, ignore_index=True)
				df_rxn_enz_pairs.loc[len(df_rxn_enz_pairs)] = [f'RXN-{rxn}_{dir}-{enz_id_final}',rxn,enz_id_final,prot_stoich_str]
				# df_rxn_enz_pairs = pd.concat([df_rxn_enz_pairs,pd.DataFrame({'id':f'RXN-{rxn}_{dir}-{enz}','enz':enz,'protein_stoich':','.join([prot+':'+str(enz_rxn_dict[enz]['protein_stoich'][prot]) for prot in enz_rxn_dict[enz]['protein_stoich']])})], ignore_index=True)

# sort by ID
df_rxn_enz_pairs = df_rxn_enz_pairs.sort_values(by=list(reversed(['protein_stoich','enz','id'])))
# remove duplicate rows
df_rxn_enz_pairs = df_rxn_enz_pairs.drop_duplicates()
df_rxn_enz_pairs.to_csv('./model/ENZYME_rxn_pairs.tsv', sep='\t', index=False)

FACOAL240_en
Updating lumpFACS180_c with {'enz': 'JL09_g3954JL09_g50'}
Updating ARABDt_c with {'gene_reaction_rule': ''}
Updating EX_arab__D_e with {'gene_reaction_rule': ''}
Updating EX_csn_e with {'gene_reaction_rule': ''}
Updating CYTK1_c with {'enz': 'JL09_g2829'}
Updating EX_cytd_e with {'gene_reaction_rule': ''}
Updating COAt_c_x with {'gene_reaction_rule': ''}
Updating EX_fru_e with {'gene_reaction_rule': ''}
Updating E4Pt_c_m with {'gene_reaction_rule': ''}
Updating EX_nmn_e with {'gene_reaction_rule': ''}
Updating EX_nac_e with {'gene_reaction_rule': ''}
Updating NCAMt_c_n with {'gene_reaction_rule': ''}
Updating NH4t_c_m with {'gene_reaction_rule': ''}
Updating NADPHt_c_r with {'gene_reaction_rule': ''}
Updating NADPt_c_r with {'gene_reaction_rule': ''}
Updating EX_dad_2_e with {'gene_reaction_rule': ''}
Updating MI1Pt_c_r with {'gene_reaction_rule': ''}
Updating MI13456Pt_c_n with {'gene_reaction_rule': ''}
Updating 2DDA7Pt_c_m with {'gene_reaction_rule': ''}
Updating EX_dur

In [11]:
# calculate MW of enzymes = sum of MW of all proteins in protein_stoich (MW from df_pro) * number of protein_stoich

# if df_enz isn't defined, create it from df_rxn_enz_pairs
if 'df_enz' not in locals():
	df_enz = df_rxn_enz_pairs.copy()
	df_enz['status'] = 'automatically added'

# if there are any rows in df_enz with 'dir' that's empty, create 2 new rows for each of those, one with 'dir' as 'FWD' and one with 'dir' as 'REV' (for faster writing to model)
if 'dir' in df_enz.columns:
	if df_enz['dir'].isnull().any():
		print('Enzyme stoichiometry has empty "dir" values. Creating FWD and REV entries for those enzymes.')
		df_enz_fwd = df_enz.copy()
		df_enz_fwd['dir'] = df_enz_fwd['dir'].fillna('FWD')
		df_enz_rev = df_enz.copy()
		df_enz_rev['dir'] = df_enz_rev['dir'].fillna('REV')
		df_enz = pd.concat([df_enz_fwd[df_enz_fwd['dir'] == 'FWD'], df_enz_rev[df_enz_rev['dir'] == 'REV']], ignore_index=True)
		# sort df_enz by 'rxn_src' and 'dir'
		df_enz = df_enz.sort_values(by=['rxn_src', 'dir']).reset_index(drop=True)
		# Now df_enz contains all original rows, plus FWD and REV for those missing 'dir'
	df_enz['id'] = df_enz.apply(lambda x: 'RXN-' + x['rxn_src'] + '_' + x['dir'] + '-' + x['enz'], axis=1)

In [12]:
for enz in df_enz.index:
	enz_mw = 0
	# robustly handle case where df_enz.at[enz, 'protein_stoich'] may be a scalar or a Series (duplicate index labels)
	ps_entry = df_enz.at[enz, 'protein_stoich']
	# normalize to list of entries
	if isinstance(ps_entry, pd.Series):
		pro_entries = ps_entry.tolist()
	else:
		pro_entries = [ps_entry]

	for ps in pro_entries:
		if pd.isna(ps):
			continue
		# each ps may contain comma-separated prot:stoich pairs
		for pro in str(ps).split(','):
			# find protein stoichiometry in df_pro
			delim = ':'
			if delim in pro:
				prot_id = pro.split(delim)[0].strip()
				try:
					prot_stoich = float(pro.split(delim)[1].strip())
				except Exception:
					prot_stoich = 1.0

				pro_matches = df_pro.loc[df_pro.gene_src == prot_id]
				# if no matches, check if prot_id is in df_pro.index
				if getattr(pro_matches, "empty", False):
					if prot_id in df_pro.index:
						# ensure pro_matches is a DataFrame for consistent access
						pro_matches = df_pro.loc[[prot_id]]
					else:
						warn(f'{prot_id} not found in protein stoichiometry file')
						continue

				# get MW value (may be Series if multiple rows)
				pro_mw = pro_matches[mw]
				if isinstance(pro_mw, pd.Series):
					if pro_mw.nunique() > 1:
						warn(f'{prot_id} has multiple MWs in df_pro')
					pro_mw = pro_mw.iloc[0]
				try:
					pro_mw = float(pro_mw)
				except Exception:
					warn(f'MW for {prot_id} is not numeric in protein stoichiometry file')
					continue

				if pro_mw == 0:
					warn(f'MW for {prot_id} is 0 in protein stoichiometry file')
					continue
				enz_mw += pro_mw * prot_stoich

	# set enz_mw to NaN if it is 0
	if enz_mw == 0:
		enz_mw = float('nan')
	df_enz.at[enz, mw] = enz_mw

# remove any duplicate rows in df_enz
df_enz = df_enz.drop_duplicates()

# move 'id' column to the front
df_enz[['id'] + [col for col in df_enz.columns if col != 'id']].to_csv('./model/ENZYME_stoich_curation.tsv', sep='\t', index=False)

In [13]:
c = -1
medium = []

### Metabolic network reaction
# Exchange reactions
for rxn in model.reactions:
	# check if rxn is in df_enz
	# if rxn.id not in df_enz.rxn_src.to_list():
	# if name matches RBA_options.exchange_regex, then it is an exchange reaction
	if re.match(RBA_defaults.exchange_regex, rxn.id):
		enz_id = 'SPONT'
		make_reversible = True
# else:
# 	gpr = rxn.gene_reaction_rule if rxn.gene_reaction_rule != "" else 'UNKNOWN'

		protein_stoich = 'zeroCost' if enz_id in RBA_defaults.spont_GPRs else ''
		
		met = [i for i in rxn.metabolites.keys()][0]
		c += 1
		tag = 'RXN'
		dir = 'FWD'
		new_id = f'{tag}-{rxn.id}_{dir}-{enz_id}'
		df_eqn.at[c, 'id'] = new_id
		df_eqn.at[c, 'type'] = 'metabolic'
		df_eqn.at[c, 'reaction'] = 'MET-' + met.id + ' -->'
		# df_eqn.loc[c, ['FBA_name','dir']] = [rxn.id,'FWD']
		df_eqn.loc[c, ['tag','FBA_name','dir','enz_id']] = [tag,rxn.id,dir,enz_id]
		
		c += 1
		dir = 'REV'
		new_id = f'{tag}-{rxn.id}_{dir}-{enz_id}'
		df_eqn.at[c, 'id'] = new_id
		df_eqn.at[c, 'type'] = 'metabolic'
		df_eqn.at[c, 'reaction'] = '-->' + 'MET-' + met.id
		# df_eqn.loc[c, ['FBA_name','dir']] = [rxn.id,'REV']
		df_eqn.loc[c, ['tag','FBA_name','dir','enz_id']] = [tag,rxn.id,dir,enz_id]

# Reactions that are not exchange reactions
enzymatic_rxns = [] # all enzyme-catalyzed rxns, not including transport rxns
enzload_rxn_dict = {}
for i in df_enz.index:
	rxn_id = df_enz.id[i]
	tag,rxn_base_id,rxn_dir,enz_id = extract_details_from_rxnid(rxn_id)

	new_id = f"{tag}-{rxn_base_id}_{rxn_dir}"
	enzload_id = f"ENZLOAD-{rxn_base_id}_{rxn_dir}-{enz_id}"
	if df_enz.protein_stoich[i] != 'zeroCost':
		enzload_rxn_dict[enzload_id] = new_id
	if new_id in df_eqn['id'].to_list():
		continue
	
	if re.match(RBA_defaults.exchange_regex, rxn_base_id):
		continue

	c += 1
	rxn_base = model.reactions.get_by_id(rxn_base_id)
	
	met_dict = metabolites_dict_from_reaction_equation_RBA(rxn_base.reaction)
	met_dict = {k:v for k,v in met_dict.items() if k != ''}
	met_dict = {'MET-' + k:v for k,v in met_dict.items()}
	if rxn_dir == 'REV':
		met_dict = {k:-v for k,v in met_dict.items()}
	elif rxn_dir == 'FWD':
		None
	else:
		print("Unknown ID that indicate reaction direction, only accepting 'FWD' and 'REV'")
	
	if enz_id not in RBA_defaults.spont_GPRs:
		df_eqn.at[c, 'coupling_type'] = 'rxn_enz'
		df_eqn.at[c, 'coupling_species'] = enz_id

	df_eqn.at[c, 'id'] = new_id
	df_eqn.at[c, 'type'] = 'metabolic'
	df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(met_dict, arrow='-->')
	df_eqn.loc[c, ['tag','FBA_name','dir','enz_id']] = [tag,rxn_base_id,rxn_dir,enz_id]
	
### Enzyme synthesis network reaction
enz_stoich = OrderedDict()
for i in df_enz.index:
	enz_stoich.setdefault(df_enz.enz[i], set()).add(df_enz.protein_stoich[i])

c = df_eqn.shape[0] - 1
for enz_id in enz_stoich.keys():
	for i,prot_str in enumerate(sorted(enz_stoich[enz_id])):
		if prot_str == 'zeroCost':
			continue
		
		c += 1
		prot_str = prot_str.split(',')
		# some stoichiometries are stored like 'JL09_g4590:1.0' -> convert via float then to int
		coeffs = OrderedDict()
		for entry in prot_str:
			parts = entry.split(':')
			prot = parts[0]
			# guard against non-numeric or missing stoich; default to 1
			try:
				st = int(float(parts[1]))
			except Exception:
				warn(f'invalid stoichiometry for {prot} in enzyme {enz_id}. Defaulting to 1.')
				st = 1
			coeffs[f'{ptmpro_prefix}-{prot}'] = -st
		coeffs[f'ENZ-{enz_id}'] = 1
		
		# differentiate enzymes with the same proteins but different stoichiometries by adding a suffix to the enzyme ID
		enzsyn_id = enz_id + '_' + str(i+1) if i > 0 else enz_id
		df_eqn.at[c, 'id'] = 'ENZSYN-' + enzsyn_id
		df_eqn.at[c, 'type'] = 'enzyme'
		df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(coeffs, arrow='-->')

#these ENZLOAD equations also need to have their MW listed in the enz_mw_g_per_mmol file
#initialize an enzload sting
enzload_mw = []
for i in df_enz.index:
	if df_enz.protein_stoich[i] == 'zeroCost':
		continue
		
	c += 1
	coeffs = OrderedDict()
	coeffs[f'ENZ-{df_enz.enz[i]}'] = -1

	enzload_id = 'ENZLOAD-' + df_enz.id[i][4:]
	#for debugging, comment out if unused
	#print("i: ",i,"\tenzload_id: ", enzload_id)
	
	df_eqn.at[c, 'id'] = enzload_id
	df_eqn.at[c, 'type'] = 'enzymeRxnLoad'
	df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(coeffs, arrow='-->')

	#get the associated mw
	enz_mw = getattr(df_enz,"MW (g/mmol)")[i]

	new_line = enzload_id + "\t" + str(enz_mw)
	if new_line not in enzload_mw:
		enzload_mw.append(new_line)

# write the file resulting from enzload_str
with open('./model/enz_mw_g_per_mmol.txt', 'w') as f:
	#TODO: update to account for impact of PTMs on enzyme MW, which may require adding separate enzymes for each combo of PTMs for each subunit. Not crucial for anything other than output reporting, but may be worth changing.
	f.write('\n'.join(enzload_mw))

### Ribosome
ribo_subunits = {}
c = df_eqn.shape[0] - 1

for rna in df_rnas.index:
	c += 1
	rna_stoich = OrderedDict({i:0 for i in ['MET-'+rna, 'MET-atp_c', 'MET-ctp_c',
											'MET-gtp_c', 'MET-utp_c', 'MET-ppi_c']})
	rna_stoich['RIBO-'+rna] = 1
	rna_stoich['MET-atp_c'] = -int(df_rnas.A[rna])
	rna_stoich['MET-ctp_c'] = -int(df_rnas.C[rna])
	rna_stoich['MET-gtp_c'] = -int(df_rnas.G[rna])
	rna_stoich['MET-utp_c'] = -int(df_rnas.U[rna])
	rna_stoich['MET-ppi_c'] = int(df_rnas.loc[rna, ['A','C','G','U']].sum())
	rna_stoich['BIO-rrna'] = df_rnas.at[rna, 'MW (g/mmol)']
	
	df_eqn.at[c, 'id'] = 'RIBOSYN-' + rna
	df_eqn.at[c, 'type'] = 'ribosome'
	df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(rna_stoich, arrow='-->')
	gene_expression_dict.setdefault(rna, []).append(df_eqn.at[c, 'id'])
with open('./model/prosyn_ribo_subunits.txt', 'w') as f, open('./model/pro_ribo_subunits.txt', 'w') as f2, open('./model/kribo.txt', 'w') as f4, open('./model/j_ribo.txt', 'w') as f5, open('./model/ribosomes.txt', 'w') as f6:
	for file in [f,f2,f4,f5,f6]: file.write('/')
	f1_lines, f2_lines = set(), set()
	for loc, ribo in ribo_dict.items():
		## proteins into ribosome subunits (to account for paralogs)
		ribo_stoich = OrderedDict()
		# if only 1 ribosome is present, all rRNAs are used
		rnas = [item for item in df_rnas.index if item.endswith(loc) or len(ribo_dict.keys()) == 1]
		ribo_df = ribo['subunits']
		# assume 1 of each rRNA and protein is used per ribosome
		for i in ribo_df.index:
			n = ribo_df.get("n", {}).get(i, 1) # default to 1 if 'n' column or value is missing
			if ribo_df.id[i] in ribo_subunits:
				ribo_subunits[ribo_df.id[i]].append(loc[1:])
			else:
				ribo_subunits[ribo_df.id[i]] = [loc[1:]]
			if ribo_df.id[i] in rnas:
				ribo_stoich['RIBO-' + ribo_df.id[i]] = -n
			else:
				# write line to f if it's not already present:
				li = "\n'"+loc[1:]+"'.'PROSYN-"+ribo_df.id[i]+"'"
				if li not in f1_lines:
					f.write(li)
					f1_lines.add(li)
				li = "\n'"+loc[1:]+"'.'"+ribo_df.id[i]+"'"
				if li not in f2_lines:
					f2.write(li)
					f2_lines.add(li)
				# if it's not a paralog of another protein, all paralogs between it and the next non-paralog are treated as paralogs
				if pd.isnull(ribo_df.paralog[i]):
					ribosub_name = 'RIBOSUB-' + ribo_df.id[i]
				
				subunit = met_ID.from_name(ribo_df.id[i]) if '-' in ribo_df.id[i] else met_ID.from_name(f'{ptmpro_prefix}-{ribo_df.id[i]}')
				prot_matches = df_pro[df_pro.index == str(subunit.base_id)]
				# if subunit.compartment, then filter prot_matches to only those with matching subloc_assigned
				if prot_matches.empty:
					prot_matches = df_pro[df_pro.gene_src == str(subunit.base_id)]
				if subunit.compartment != None:
					prot_matches = prot_matches[prot_matches['subloc_assigned'].apply(lambda x: x == subunit.compartment if pd.notna(x) else False)]
				else:
					# filter by entries where the compartment matches the ribosome's location
					prot_matches = prot_matches[prot_matches['subloc_assigned'].apply(lambda x: x == loc[1:] if pd.notna(x) else False)]
				if prot_matches.empty:
					warn(f'no matches found in df_pro for ribosome subunit {ribo_df.id[i]} with base ID {subunit.base_id} and compartment {subunit.compartment}. This subunit will be skipped.')
					continue
				# add rxn converting the protein into the ribosome subunit (RIBOSUB prefix)
				for iter, (_, match) in enumerate(prot_matches.iterrows()):
					c += 1
					ribosub_stoich = OrderedDict()
					ribosub_stoich[f'{ptmpro_prefix}-{match["id"]}'] = -1
					ribosub_stoich[ribosub_name] = 1
					eq_id = f'RIBOSUBSYN-{loc[1:]}-{ribosub_name}-FROM-{ribo_df.id[i]}'
					eq_id = f'{eq_id}-{iter+1}' if iter > 0 else eq_id
					while eq_id in df_eqn.id.to_list():
						eq_id += '-DUP'
					df_eqn.at[c, 'id'] = eq_id
					df_eqn.at[c, 'type'] = 'ribosome-subunit'
					df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(ribosub_stoich, arrow='-->')
				# add the ribosome subunit to the ribosome
				ribo_stoich[ribosub_name] = -n
		# making the ribosome itself from the subunits
		c += 1
		df_eqn.at[c, 'id'] = 'RIBOSYN-ribonuc' if loc == '_c' else 'RIBOSYN-ribomito' if loc == '_m' else 'RIBOSYN-ribo'+loc
		# add to ribosome files
		ribo_id = "'"+loc[1:]+"'"
		f6.write('\n'+ribo_id); f5.write('\n'+ribo_id+".'"+df_eqn.at[c, 'id']+"'"); f4.write('\n'+ribo_id+' '+str(ribo.get('k_ribo', RBA_defaults.k_ribo_max)))
		df_eqn.at[c, 'type'] = 'ribosome'
		df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(ribo_stoich, arrow='-->')
	for file in [f,f2,f4,f5,f6]: file.write('\n/')
# write the ribosome subunit stoichiometry to a json file
with open('./model/ribo_subunits.json', 'w') as f:
	json.dump(ribo_subunits, f)

#### Build files related to MW

In [14]:
data = getattr(df_pro,'MW (g/mmol)').to_dict()
ids = getattr(df_pro,'id').to_list()

#build the string to write
prot_mw_str = ""

#for each id
for id in ids:
	new_line = id + "\t" + str(data[id]) + "\n"
	prot_mw_str = prot_mw_str + new_line
# add dummy proteins to the string
for prot in dummy_list:
	new_line = prot["name"].replace('PROSYN-','',1) + "\t" + str(prot["MW"]) + "\n"
	prot_mw_str = prot_mw_str + new_line

#write to output
with open('./model/pro_mw_g_per_mmol.txt', 'w') as f:
	f.write(prot_mw_str)

In [15]:
#TEST: this whole block. get MW list
mw_list = getattr(df_enz,"MW (g/mmol)").to_list()

#get enz list
enz_list = getattr(df_enz,"enz").to_list()

#create string for writing output file
enz_mw_str = "enz_id\tMW (g/mmol)\n"

#add to the string for enzyme entries
#need to initialize a counter
c = 0

#list to keep track of existing entries
#predefined values prevent adding unknown
existing_pro = ['nan','unknown']

for enz in enz_list:

	#only add a new line if unique entry
	if not(str(enz) in existing_pro):

		#for debugging, comment out if unused
		#print("enz: |",enz,"|, type: ",type(enz))
	
		new_line = str(enz) + "\t" + str(mw_list[c]) + "\n"

		#for debugging, comment out if unused
		#print(new_line)

		enz_mw_str = enz_mw_str + new_line

		existing_pro.append(enz)

	c = c + 1

#write to output
with open('./model/enz_mw_g_per_mmol_norxnmapped.txt', 'w') as f:
	f.write(enz_mw_str)

#### Protein file

In [ ]:
### Protein

pro_lengths = {}

c = df_eqn.shape[0] - 1
translation_loci = set()
unknown_ribo_prosyn = set()
protein_functions = {}

def edit_stoich(stoich:OrderedDict,edits):
	# Handle case where edits might be a Series - extract the scalar value
	if isinstance(edits, pd.Series):
		edits = edits.iloc[0] if len(edits) > 0 else None
	if pd.notna(edits):
		cofs_st = edits.split(',')
		cofs_st = OrderedDict({c.split(':')[0]:c.split(':')[1] for c in cofs_st})
		for k,v in cofs_st.items():
			if float(v) == 0:
				continue
			stoich['MET-'+k] = -int(v)
	return stoich

# make dict version with all keys and values from enz_rxn_dict, but only for transport enzymes, and with an added key for surface area initialized to 0
transport_enzymes = {enzyme: {**enz_rxn_dict[enzyme], 'surface_area': 0} for enzyme in enz_rxn_dict if 'transport' in [model.reactions.get_by_id(rxn).subsystem.lower() for rxn in enz_rxn_dict[enzyme]['rxns'] if rxn in model.reactions] or any(rxn_type(model.reactions.get_by_id(rxn)) == 'transport' for rxn in enz_rxn_dict[enzyme]['rxns'] if rxn in model.reactions)}
#TODO: fix this so it doesn't include excluded rxns
with open('./model/transport_rxns.txt', 'w') as f:
	for rxn in sorted(set(rxn for details in transport_enzymes.values() for rxn in details["rxns"])):
		f.write(f"'{rxn}'\n")
transport_enzyme_SA_found = False
bio_prot_types = []
# Protein synthesis/waste reactions
c = df_eqn.shape[0] - 1

eqns_in_dict_format = {row['id']:metabolites_dict_from_reaction_equation_RBA(row['reaction'], metabolites_to_ignore=['']) for index, row in df_eqn.iterrows()}
eqns_to_consider = {k:{k2:v2 for k2,v2 in v.items() if k2.startswith(ptmpro_prefix+'-')} for k,v in eqns_in_dict_format.items() if not any(k.startswith(prefix) for prefix in ['PTM-','PTMWASTE-'])}
eqns_to_consider = {k:v for k,v in eqns_to_consider.items() if v} # remove any empty dicts

pro_location_map = {}
ptm_protein_map = {}
protein_unique_ptm_map = {} # for mapping PTMs that aren't used outside PTMWASTE but provide unique metabolites (not counting compartment differences)
protein_unique_ptm_and_used_map = {} 
with open('./model/pro_syn_waste.txt', 'w') as f, open('./model/protein_membrane_surface_area.txt', 'w') as surface_area_f:
	f.write('/')
	for i in df_pro.index: 
		c += 1
		pro_id = df_pro.at[i, 'id'] if 'id' in df_pro.columns else i
		prepro = 'PRO-' + str(pro_id) # preprotein (from translation)
		pro_id_parts = met_ID.from_name(prepro)
		# update prepro to keep everything except PTMs
		pro_id_parts_dict = vars(pro_id_parts)
		pro_id_parts_dict.pop('PTMs', None)
		if 'compartment' not in pro_id_parts_dict:
			pro_id_parts_dict['compartment'] = None
		prepro = str(met_ID(**pro_id_parts_dict))
		prot = f'{ptmpro_prefix}-{str(pro_id)}' # protein after post-translational modifications (PTMs)
		prosyn_met_id = prepro.split('PRO-')[1] # ID of the metabolite used in PROSYN rxn
		prosyn_id = 'PROSYN-' + prosyn_met_id
		bio_prot_type = 'BIO-prot'
		total_membrane_area = 0
		keywords = set(df_pro.at[i, 'Keywords'].strip().split(';')) if 'Keywords' in df_pro.columns and pd.notna(df_pro.at[i, 'Keywords']) else set()
		# print(f'Building translation reaction for {prosyn_id} from protein {df_pro.id[i]}')
		for col in ['Transmembrane', 'Intramembrane']:
			if pd.notna(df_pro.get(col, {}).get(i, pd.NA)):
				for region in re.split(r'\W*'+col.upper().replace('BRANE','')+' ', df_pro.at[i, col]):
					region_type_search = re.search(r' /note="*([^"]*)"*', region)
					if region_type_search:
						region_type = region_type_search.group(1)
						if 'helical' in region_type.lower():
							if col == 'Transmembrane':
								total_membrane_area += .5
							elif col == 'Intramembrane':
								total_membrane_area += .25 # since it doesn't go through the whole membrane, we assume it occupies half the membrane surface area of a fully transmembrane region. This is a simplification and may be updated in the future as more information about intramembrane regions is gathered.
						else:
							warn(f'Unknown {col} region type "{region_type}" for protein {df_pro.id[i]}. Currently only "helical" regions are supported.', ignore_duplicate_warnings=False)
		if total_membrane_area == 0 and 'Transmembrane helix' in keywords: # only use for proteins w/ no surface area to avoid accidentally double-counting helices
			total_membrane_area += .5 * ((1.868/2.274) ** 2) # transmembrane alpha-helix dimensions multiplied by ratio of 3_10 helix radius (helix type with smallest radius) to alpha-helix radius, from https://link.springer.com/article/10.1007/s00894-013-1819-7/tables/1 (used ratio in case the methods differ from the alpha-helix estimates I used). Squared since we're calculating area.
		if total_membrane_area > 0:
			bio_prot_type = 'BIO-protmem'
			# add total_membrane_area to transport_enzymes if this protein is in a transport enzyme, to account for the fact that all proteins in a transport enzyme will contribute to the surface area occupied by the transport enzyme
			for enzyme, details in transport_enzymes.items():
				for id in [prosyn_met_id, prot, pro_id, df_pro.id_without_compartment[i]]:
					if id in details.get('protein_stoich', {}):
						details['surface_area'] += total_membrane_area * details['protein_stoich'][id]
						transport_enzyme_SA_found = True
						break
		bio_prot_type += '_' + df_pro.subloc_assigned[i]
		biosyn_id = 'BIOSYN-' + bio_prot_type.split('-',1)[-1]
		if biosyn_id not in df_eqn.id.to_list():
			# add to list by converting bio_prot_type to BIOSYN-PROTMODELED
			df_eqn.at[c, 'id'] = biosyn_id
			df_eqn.at[c, 'type'] = 'biomass'
			df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA({bio_prot_type: -1, 'BIO-protmodeled': 1}, arrow='-->')
			df_eqn.at[c, 'compartment'] = df_pro.subloc_assigned[i]
			bio_prot_types.append(biosyn_id)
			c += 1
		trans_st = OrderedDict()
		PTM_st = OrderedDict({prepro:-1,prot:1})
		for met in ['MET-atp_c', 'MET-h2o_c',
					'MET-adp_c', 'MET-pi_c', 'MET-h_c', 'MET-gtp_c',
					'MET-gdp_c']:
			trans_st[met] = 0
		seq = df_pro.sequence[i].replace("*","")
		pro_lengths[prosyn_met_id] = len(seq)
		# add PTM reaction only when PTM edits exist or the modified protein is utilized
		ptm_edits = df_pro.get('PTM_stoich_edits', pd.Series(index=df_pro.index, dtype=object)).get(i, pd.NA)
		PTM_st = edit_stoich(PTM_st, ptm_edits)
		has_ptm_edits = len(PTM_st) > 2 # if there are more than just the prepro and prot in PTM_st, then there are PTM edits

		# if protein isn't already in df_eqn...
		prowaste_rxns_to_add = {}
		if prosyn_id not in df_eqn.id.to_list():
			prowaste_rxns_to_add[''] = 'PROSYN'
			if 'trans_stoich_edits' in df_pro.columns:
				trans_st = edit_stoich(trans_st,df_pro.trans_stoich_edits[i])

			for aa in df_aamap.index:
				trans_st[df_aamap.tRNA_in[aa]] = -seq.count(aa)
				trans_st[df_aamap.tRNA_out[aa]] = seq.count(aa)
						
			trans_st[prepro] = 1
			pro_location_map[prepro] = bio_prot_type
			df_eqn.at[c, 'coupling_type'] = 'prot_ribo'
			df_eqn.at[c, 'coupling_species'] = 'ribo'
			
			# protein will occupy cellular space in its specific compartment
			trans_st[bio_prot_type] = trans_st.get(bio_prot_type, 0) + df_pro.at[i, 'MW (g/mmol)']
			
			# Cost: Initiation: 1 ATP + 2 GTP (initiate and bind Methionine)
			for met in ['MET-atp_c', 'MET-h2o_c']:
				trans_st[met] -= 1
			for met in ['MET-adp_c', 'MET-pi_c', 'MET-h_c']:
				trans_st[met] += 1
						
			# Elongation: 2 GTP / cycle (2(n-1) cycles for n amino acids + 2 GTP from initiation = 2n GTP consumed so far)
			# (elongation process excludes Methionine since it is already bound in initiation process)
			# Termination and recycling: 1 GTP + 1 ATP: https://doi.org/10.1534/genetics.115.186221
			# 	However, we exclude these costs since it's not always required (e.g., via "leaky scanning"): https://doi.org/10.1093/nar/gkp765
			for met in ['MET-gtp_c', 'MET-h2o_c']:
				trans_st[met] -= 2*len(seq)
			for met in ['MET-gdp_c', 'MET-pi_c', 'MET-h_c']:
				trans_st[met] += 2*len(seq)
			
			df_eqn.at[c, 'id'] = prosyn_id

			# add translation rxn to gene_expression_dict list, or create a new list if it doesn't exist
			gene_expression_dict.setdefault(df_pro.gene_src[i], []).append(df_eqn.at[c, 'id'])

			# add to protein functions dictionary under its compartment-free ID
			protein_functions.setdefault(df_pro.id_without_compartment[i], {})[prosyn_met_id] = {}
			df_eqn.at[c, 'type'] = 'protein'
			df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(trans_st, arrow='-->')
			if total_membrane_area > 0:
				surface_area_f.write(f"SA('{prosyn_id}') = {total_membrane_area} * pi;\n")
			df_eqn.at[c, 'compartment'] = df_pro.subloc_assigned[i]
			# Set ribosome translating the protein
			if df_pro.at[i, 'translation_loc'] in ['unknown','any']:
				unknown_ribo_prosyn.add("'"+df_eqn.at[c, 'id']+"'")
			else:
				translation_loci.add("'"+str(df_pro.at[i, 'translation_loc'])+"'.'"+df_eqn.at[c, 'id']+"'")

		if has_ptm_edits: # if there are PTM edits or the protein is used in a reaction, and there are more than just the prepro and prot in PTM_st
			c += 1
			# # PTM_st_simplified = {re.sub(f"{RBA_defaults.compartment_regex['default']}", "", k[4:]):v for k,v in PTM_st.items() if v != 0 and k.startswith('MET-')}
			# PTM_st_simplified = {k[4:]:math.copysign(v) for k,v in PTM_st.items() if v != 0 and k.startswith('MET-')}
			# # add to protein_unique_ptm_map if the PTM_st_simplified is not empty and isn't already in the map for this prepro
			# existing_ptms = protein_unique_ptm_map.get(prepro, {}).values() if prepro in protein_unique_ptm_map else []
			# if PTM_st_simplified:
			# 	if prepro not in existing_ptms:
			# 		protein_unique_ptm_map.setdefault(prepro, {}).update({prot:{PTM_st_simplified}})
			# 	if any(prot in met_dict for met_dict in eqns_to_consider.values()):
			# 		protein_unique_ptm_and_used_map.setdefault(prepro, {}).update({prot:{PTM_st_simplified}})
			# 	# elif this PTM is used in rxn in eqns_to_consider, then add it to the map for this prepro and remove any other PTMs for this prepro that aren't used in any reactions in eqns_to_consider
			# 	elif any(prot in met_dict for met_dict in eqns_to_consider.values()):
			# 		protein_unique_ptm_map.setdefault(prepro, {}).update({prot:{PTM_st_simplified}})
			# 		# remove any other PTMs for this prepro that aren't used in any reactions in eqns_to_consider
			# 		for other_prot in list(protein_unique_ptm_map[prepro].keys()):
			# 			if other_prot != prot and not any(other_prot in met_dict for met_dict in eqns_to_consider.values()):
			# 				del protein_unique_ptm_map[prepro][other_prot]

			# protein_unique_ptm_map.setdefault(prepro, {}).update({prot:{PTM_st_simplified}})
			# # Check if the protein is used in any reactions in eqns_to_consider, and if so, add a PTM reaction for it

			prowaste_rxns_to_add['PTM'] = 'PTM'
			if RBA_defaults.biomass_impacted_by_PTMs:
				# add/subtract masses of metabolites from PTM to metabolite representing protein mass
				for met,cof in PTM_st.copy().items():
					if met.startswith('MET-'):
						met_id = met.split('MET-',1)[1]
						if cof > 0:
							PTM_st[bio_prot_type] = PTM_st.get(bio_prot_type, 0) - (cof * model.metabolites.get_by_id(met_id).formula_weight/1000)
						elif cof < 0:
							PTM_st[bio_prot_type] = PTM_st.get(bio_prot_type, 0) + (-cof * model.metabolites.get_by_id(met_id).formula_weight/1000)
				pro_mw_modified = df_pro.at[i, 'MW (g/mmol)'] if pd.notna(df_pro.at[i, 'MW (g/mmol)']) else 0
				pro_mw_modified += PTM_st.get(bio_prot_type, 0)
				if bio_prot_type != pro_location_map[prepro]:
					PTM_st[bio_prot_type] += df_pro.at[i, 'MW (g/mmol)']
					PTM_st[pro_location_map[prepro]] = PTM_st.get(pro_location_map[prepro], 0) - df_pro.at[i, 'MW (g/mmol)']
			df_eqn.at[c, 'id'] = 'PTM-' + df_pro.id[i]
			df_eqn.at[c, 'type'] = 'proteinPTM'
			df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(PTM_st, arrow='-->')
			ptm_protein_map.setdefault(prot, []).append(prepro)
		else:
			for eqn_id, met_dict in eqns_to_consider.items():
				if prot in met_dict:
					# replace the protein in the reaction with its prepro version
					df_eqn.at[df_eqn.index[df_eqn['id'] == eqn_id][0], 'reaction'] = re.sub(r'\b' + re.escape(prot) + r'\b', prepro, df_eqn.at[df_eqn.index[df_eqn['id'] == eqn_id][0], 'reaction'])

		for abbv,tag in prowaste_rxns_to_add.items():
			c += 1
			pro_waste_prot_id = prosyn_met_id if abbv != 'PTM' else df_pro.id[i]
			pro=abbv+'PRO-'+pro_waste_prot_id
			new_st = OrderedDict()
			new_st[pro] = -1
			# add the protein's MW as the coefficient for BIO-protwasted
			new_st['BIO-protwasted'] = df_pro.at[i, 'MW (g/mmol)'] if abbv != 'PTM' else pro_mw_modified
			
			pw_id = f"{abbv}WASTE-{pro_waste_prot_id}" if abbv == 'PTM' else f"PROWASTE-{pro_waste_prot_id}"
			f.write(f"\n'{tag}-{pro_waste_prot_id}'.'{pw_id}'")

			df_eqn.at[c, 'id'] = pw_id
			df_eqn.at[c, 'type'] = 'proteinWaste'
			df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(new_st, arrow='-->')
	f.write('\n/')
with open('./model/bio_prot_types.txt', 'w') as f:
	f.write('\n'.join(['/'] + sorted([f"'{t}'" for t in bio_prot_types]) + ['/']))
# write the protein length file
len_str = "/\n"
for prot in pro_lengths:
	len_str = len_str + "'PROSYN-" + prot + "' " + str(pro_lengths[prot]) + "\n"
for prot in dummy_list:
	len_str += "'" + prot["name"] + "' " + str(prot["medianL"]) + "\n"
len_str = len_str + "/"
with open('./model/RBA_proteinLength.txt', 'w') as f:
	f.write(len_str)

# check if any transport_enzymes have surface area of 0, and if so, warn that they may need to be updated with surface area contributions from their protein subunits
if transport_enzyme_SA_found:
	transport_enzymes_no_SA = [enzyme for enzyme, details in transport_enzymes.items() if details['surface_area'] == 0]
	if transport_enzymes_no_SA:
		warn(f'The following transport enzymes have no surface area contributions from their protein subunits and may need to be updated: {", ".join(transport_enzymes_no_SA)}')

### Dummy protein
for prot in dummy_list:
	dummy_MW = prot["MW"]
	trans_st = make_dummy_protein_stoich(aa_standards_df=df_aamap, prot_df=prot["df"], length=prot["medianL"], rxn_name=prot["name"], mw=dummy_MW)

	c += 1
	# prot_st["BIO-protdummy"] = prot["MW"]
	df_eqn.at[c, 'id'] = prot["name"]
	df_eqn.at[c, 'coupling_type'] = 'prot_ribo'
	df_eqn.at[c, 'coupling_species'] = 'ribo' #TEST: Update for ribonuc, ribomito, and other ribo types
	df_eqn.at[c, 'type'] = 'protein'
	df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA(trans_st,
															arrow='-->', floatdecimal=6)
	if prot["translation_loc"] in ['unknown','any']:
		unknown_ribo_prosyn.add("'"+df_eqn.at[c, 'id']+"'")
	else:
		translation_loci.add("'"+df_eqn.at[c, 'id']+"'.'"+df_eqn.at[c, 'id']+"'")

# Pseudoreaction representing mass of waste protein (for kapps): PROWASTE-TOTALPROTEIN
c += 1
df_eqn.at[c, 'id'] = 'PROWASTE-TOTALPROTEIN'
df_eqn.at[c, 'type'] = 'proteinWaste'
df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA({'BIO-protwasted':-1}, arrow='-->')

for loc in ['modeled','dummy','cyt','mito']: # add BIOWASTE rxns for biomass-related byproduct of PROSYN rxns, to support simulations at 0 growth
	c += 1
	df_eqn.at[c, 'id'] = 'BIOWASTE-PROT'+loc.upper()
	df_eqn.at[c, 'type'] = 'bioWaste'
	df_eqn.at[c, 'reaction'] = build_reaction_equation_from_metabolites_dict_RBA({'BIO-prot'+loc:-1}, arrow='-->')

# save biowaste rxns to a file
with open('./model/RBA_rxns_biowaste.txt', 'w') as f:
	f.write('\n'.join(['/'] + [f"'{i}'" for i in df_eqn.id.to_list() if i.startswith('BIOWASTE-')] + ['/']))

### Write translation localization lists
with open('./model/translation.txt', 'w') as f:
	f.write('\n'.join(['/'] + sorted(list(translation_loci)) + ['/']))
with open('./model/translated_by_any_ribo.txt', 'w') as f:
	f.write('\n'.join(['/'] + sorted(list(unknown_ribo_prosyn)) + ['/']))

### Biomass
for i in df_biom.index:
	c += 1
	if not pd.isna(df_biom.rxn_id[i]):
		df_eqn.at[c, 'id'] = df_biom.rxn_id[i]
		df_eqn.at[c, 'type'] = 'biomass'
		df_eqn.at[c, 'reaction'] = df_biom.rxn_equation[i]
# make into file RBA_rxns_whole_biomass.txt
with open('./model/RBA_rxns_whole_biomass.txt', 'w') as f:
	# find whole_biomass_rxns in df_eqn and write them to file
	f.write('\n'.join(['/'] + [i for i in whole_biomass_rxns if i in df_eqn.id.to_list()] + ['/']))

In [17]:
for k,v in gene_expression_dict.items():
	# check for entries in v starting with "'RIBO"
	if any([i.startswith("'RIBO") for i in v]):
		warn(f"{k} has entries starting with 'RIBO' in gene_expression_dict")

In [18]:
# save gene list to file
with open('./model/genes.txt', 'w') as f:
	f.write('\n'.join(['/'] + [f"'{i}'" for i in genes] + ['/']))
for gene in genes:
	if gene not in gene_expression_dict:
		gene_expression_dict[gene] = []
# remove duplicates from gene_expression_dict, preserve order
gene_expression_dict = {k: list(dict.fromkeys([f"'{i}'" for i in v])) for k, v in gene_expression_dict.items()}
# save gene-protein mapping to file
with open('./model/genes_expression.txt', 'w') as f:
	f.write('\n'.join(['/'] + [f"'{gene}'.({','.join(gene_expression_dict[gene])})" for gene in genes if gene_expression_dict[gene]] + ['/']))

In [19]:
# remove all blocked metabolites from df_eqn (and their associated reactions) and save to a new file
met_coeffs = {}
blocked_mets = set()
eqns_in_dict_format = {row['id']:metabolites_dict_from_reaction_equation_RBA(row['reaction'], metabolites_to_ignore=['']) for index, row in df_eqn.iterrows()}
for id, eqn in eqns_in_dict_format.items():
	for met, coeff in eqn.items():
		met_coeffs.setdefault(met, {'rxns': [], 'coeffs': []})
		met_coeffs[met]['rxns'].append(id)
		met_coeffs[met]['coeffs'].append(coeff)
for met, data in met_coeffs.items():
	coeffs = data['coeffs']
	if all(c == 0 for c in coeffs):
		blocked_mets.add(met)
		print(f"Blocked metabolite {met} has all zero coefficients in reactions: {data['rxns']}")
	# if there aren't positive and negative coefficients for the metabolite, it's blocked
	elif not (any(c > 0 for c in coeffs) and any(c < 0 for c in coeffs)):
		blocked_mets.add(met)
		print(f"Blocked metabolite {met} does not have both positive and negative coefficients in reactions: {data['rxns']}")
# remove blocked metabolites from df_eqn
# df_eqn = df_eqn[~df_eqn['id'].isin([rxn for met in blocked_mets for rxn in met_coeffs[met]['rxns']])]

Blocked metabolite MET-mpdol_g does not have both positive and negative coefficients in reactions: ['RXN-12AMANTF_g_FWD']
Blocked metabolite MET-mmpdol_g does not have both positive and negative coefficients in reactions: ['RXN-12AMANTF_g_FWD']
Blocked metabolite MET-chito2pdol_c does not have both positive and negative coefficients in reactions: ['RXN-14BMANTF_c_FWD']
Blocked metabolite MET-mpdol_c does not have both positive and negative coefficients in reactions: ['RXN-14BMANTF_c_FWD']
Blocked metabolite MET-2mbac_c does not have both positive and negative coefficients in reactions: ['RXN-2MBACt_c_e_FWD']
Blocked metabolite MET-3hph5mb_c does not have both positive and negative coefficients in reactions: ['RXN-3DH5HPBMT_c_FWD']
Blocked metabolite MET-3hph5mb_m does not have both positive and negative coefficients in reactions: ['RXN-3DH5HPBMT_m_FWD']
Blocked metabolite MET-3hhdcoa_rm does not have both positive and negative coefficients in reactions: ['RXN-3OACR160_rm_FWD']
Blocked 

In [20]:
# check for any rxns that subtract more BIO-prot metabolites than they add
for id, eqn in eqns_in_dict_format.items():
	for met, coeff in eqn.items():
		if met.startswith('BIO-prot') and coeff < 0:
			if -coeff > sum(v for k,v in eqn.items() if k.startswith('BIO-prot') and v > 0):
				print(f"Warning: Reaction {id} subtracts more of {met} than it adds of other BIO-prot metabolites.")

#### Save excel files

In [21]:
df_eqn = df_eqn.drop_duplicates(ignore_index=True)
df_eqn.to_csv('./model/RBA_stoichiometry.tsv', sep='\t', index=None)

# Build GAMS Sij and flux bounds (originally A02_build_GAMS_Sij_and_fluxBounds.ipynb)

#### Load excel file

In [22]:
df_eqn = read_spreadsheet('./model/RBA_stoichiometry.tsv')

#### Assemble list of species

In [23]:
met_list_raw = []
for i in df_eqn.index:
	met_dict = metabolites_dict_from_reaction_equation_RBA(df_eqn.reaction[i])
	met_list_raw += met_dict.keys()

met_list = sorted(list(set(met_list_raw)))
met_list = [i for i in met_list if i != '']
with open('./model/RBA_species.txt', 'w') as f:
	f.write('\n'.join(['/'] + ["'" + i + "'" for i in met_list] + ['/']))
# add list of demand rxns for each metabolite (for testing purposes)
demand_list = ["DM-" + i for i in met_list]
#TODO: add to df_eqn
with open('./model/RBA_rxns_demand.txt', 'w') as f:
	f.write('\n'.join(['/'] + ["'DM-" + i + "'" for i in met_list] + ['/']))

#### Assemble list of reactions

In [24]:
rxn_list = df_eqn.id.to_list()
rxn_list = ["'" + i + "'" for i in rxn_list if i != '']
rxn_list = ['/'] + rxn_list + ['/']

In [25]:
with open('./model/RBA_rxns.txt', 'w') as f:
	f.write('\n'.join(rxn_list))

#### Get list of uptake reactions

In [26]:
rxn_list = []
for i in df_eqn.id:
	tag,rxn_base_id,rxn_dir,enz_id = extract_details_from_rxnid(i)
	if tag == 'RXN' and re.match(RBA_defaults.exchange_regex, rxn_base_id) and rxn_dir == 'REV':
		rxn_list.append(i)

rxn_list = ["'" + i + "'" for i in rxn_list if i != '']
rxn_list = ['/'] + rxn_list + ['/']

In [27]:
with open('./model/RBA_rxns_EXREV.txt', 'w') as f:
	f.write('\n'.join(rxn_list))

#### Get list of secretion reactions

In [28]:
rxn_list = []
for i in df_eqn.id:
	tag,rxn_base_id,rxn_dir,enz_id = extract_details_from_rxnid(i)
	if tag == 'RXN' and re.match(RBA_defaults.exchange_regex, rxn_base_id) and rxn_dir == 'FWD':
		rxn_list.append(i)

rxn_list = ["'" + i + "'" for i in rxn_list if i != '']
rxn_list = ['/'] + rxn_list + ['/']

In [29]:
with open('./model/RBA_rxns_EXFWD.txt', 'w') as f:
	f.write('\n'.join(rxn_list))

#### Get list of protein waste reactions

In [30]:
rxn_list = []
ptmwaste_list = []
for i in df_eqn.id:
	tag = i.split('-')[0]
	if tag in ['PROWASTE',f'PTMWASTE']:
		rxn_list.append(i)
		if tag == 'PTMWASTE':
			ptmwaste_list.append("'"+i+"'")

rxn_list = ["'" + i + "'" for i in rxn_list if i != '']
rxn_list = ['/'] + rxn_list + ['/']

with open('./model/RBA_rxns_prowaste.txt', 'w') as f:
	f.write('\n'.join(rxn_list))
with open('./model/RBA_rxns_ptmwaste.txt', 'w') as f:
	f.write('\n'.join(['/'] + ptmwaste_list + ['/']))

#### Get list of enzyme syn reactions

In [31]:
rxn_list = []
for i in df_eqn.id:
	tag = i.split('-')[0]
	if tag == 'ENZSYN':
		rxn_list.append(i)

rxn_list = ["'" + i + "'" for i in rxn_list if i != '']
rxn_list = ['/'] + rxn_list + ['/']

In [32]:
with open('./model/RBA_rxns_enzsyn.txt', 'w') as f:
	f.write('\n'.join(rxn_list))

#### Get list of enzyme load reactions

In [33]:
enzloads = ["'" + i + "'" for i in sorted(set(enzload_rxn_dict.keys())) if i != '']
with open('./model/RBA_rxns_with_enz.txt', 'w') as f:
	f.write('\n'.join(['/'] + ["'" + i + "'" for i in sorted(set(enzload_rxn_dict.values())) if i != ''] + ['/']))
# list created so GAMS knows which ENZLOAD corresponds to which rxn
with open('./model/RBA_rxns_enzload.txt', 'w') as f:
	f.write('\n'.join(['/'] + enzloads + ['/']))
with open('./model/RBA_rxn_enzload_coupling.txt', 'w') as f:
	f.write('\n'.join(['/'] + [f"'{k}'.'{v.replace('ENZLOAD-','RXN-',1)}' 1" for k, v in enzload_rxn_dict.items() if k != ''] + ['/']))

#### Get list of metabolic network reactions

In [34]:
rxn_list = []
for i in df_eqn.id:
	tag = i.split('-')[0]
	if tag == 'RXN':
		rxn_list.append(i)

rxn_list = ["'" + i + "'" for i in rxn_list if i != '']
rxn_list = ['/'] + rxn_list + ['/']

In [35]:
with open('./model/RBA_rxns_rxnmetabolicnetwork.txt', 'w') as f:
	f.write('\n'.join(rxn_list))

In [36]:
# Make k_app (enzyme turnover rate) file draft using placeholder values
kapp_list = []
kapp_list = [i + " 360000" for i in enzloads if i != '/']
kapp_list = ['/'] + kapp_list + ['/']
with open('./model/kapp_placeholders.txt', 'w') as f:
	f.write('\n'.join(kapp_list))

#### Assemble stoichiometry

In [37]:
# make extra kapp calculation files
directory = '../GAMS/parameterization/enz_from_proteome/'
if not os.path.exists(directory):
	os.makedirs(directory)

sij = []
rxns_enz = []
rxns_proenz = []
sij_proenz = []
i_proenz = set()
i_pro = set()
i_ptmpro = set()

for i in df_eqn.index:
	x = metabolites_dict_from_reaction_equation_RBA(df_eqn.reaction[i])
	met_dict = {}
	for k,v in x.items():
		if k == '':
			continue
		# note when a protein is used in a reaction other than its PROWASTE and production reactions
		met = k.split('-',1)[1]
		original_pro = None
		if k.startswith(ptmpro_prefix + '-') and not any(df_eqn.id[i].startswith(prefix) for prefix in ['PTMWASTE-', 'PTM-']):
			original_pro = ptm_protein_map.get(k, [met])[0].split('-',1)[1] if ptm_protein_map.get(k, [met]) else met.split('-',1)[1]
		elif k.startswith('PRO-') and not any(df_eqn.id[i].startswith(prefix) for prefix in ['PROWASTE-', 'PROSYN-', 'PTM-']):
			original_pro = met
		if original_pro:
			# add the protein function to the protein_functions dictionary under its compartment-free ID
			for gene,prots in protein_functions.items():
				if met in prots:
					protein_functions[gene][met].update({df_eqn.id[i]:v})
					break
				elif original_pro in prots:
					protein_functions[gene][original_pro].update({df_eqn.id[i]:v})
					break
				elif gene == original_pro:
					protein_functions[gene][original_pro].update({df_eqn.id[i]:v})
					break
		if v.is_integer():
			met_dict[k] = int(v)
		else:
			met_dict[k] = v
			
	rxn_prefixes = ('ENZ', 'PTM-', 'PROSYN-')
	if df_eqn.id[i].startswith(rxn_prefixes):
		id_for_efp = df_eqn.id[i]
		if df_eqn.id[i].startswith('PROSYN-'):
			id_for_efp = id_for_efp.replace('PROSYN-','PROIN-',1)
		elif df_eqn.id[i].startswith('ENZ'):
			rxns_enz.append(f"'{df_eqn.id[i]}'")
		rxns_proenz.append(f"'{id_for_efp}'")
		# sij.append("'" + df_eqn.id[i] + "'.'" + df_eqn.id[i] + "' 1")
		for k,v in met_dict.items():
			sij.append("'" + k + "'.'" + df_eqn.id[i] + "' " + str(v))
			if any(k.startswith(prefix) for prefix in (ptmpro_prefix + '-', 'PRO-', 'ENZ-')):
				sij_proenz.append("'" + k + "'.'" + id_for_efp + "' " + str(v))
				i_proenz.add(k)
				if k.startswith('PRO-'):
					i_pro.add(k)
				elif k.startswith(ptmpro_prefix + '-'):
					i_ptmpro.add(k)
	else:
		for k,v in met_dict.items():
			sij.append("'" + k + "'.'" + df_eqn.id[i] + "' " + str(v))
	
sij = ['/'] + sij + ['/']
with open('./model/RBA_sij.txt', 'w') as f:
	f.write('\n'.join(sij))
# write protein functions to file
with open('./protein_functions.json', 'w') as f:
	json.dump(protein_functions, f, indent=4)
# make set of all proteins in protein_functions with no modeled functions
proteins_without_modeled_functions = set()
# identify them by checking if all values within values are empty
for gene,prots in protein_functions.items():
	function_found = False
	for prot,funcs in prots.items():
		# if all values are empty, add to set
		if funcs.values():
			function_found = True
			break
	if not function_found:
		proteins_without_modeled_functions.add(gene)
with open('./proteins_without_modeled_functions.txt', 'w') as f:
	f.write('\n'.join(sorted(list(f"'{p}'" for p in proteins_without_modeled_functions))))
with open(directory + 'rxns_enz.txt', 'w') as f:
	f.write('\n'.join(['/'] + sorted(rxns_enz) + ['/']))
with open(directory + 'rxns_pro_and_enz.txt', 'w') as f:
	f.write('\n'.join(['/'] + sorted(rxns_proenz) + ['/']))
with open(directory + 'sij_pro_and_enz.txt', 'w') as f:
	f.write('\n'.join(['/'] + sij_proenz + ['/']))
with open(directory + 'pro_and_enz.txt', 'w') as f:
	f.write('\n'.join(['/'] + sorted(list(f"'{p}'" for p in i_proenz)) + ['/']))
with open('./model/pro.txt', 'w') as f:
	f.write('\n'.join(['/'] + sorted(list(f"'{p}'" for p in i_pro)) + ['/']))
with open('./model/ptmpro.txt', 'w') as f:
	f.write('\n'.join(['/'] + sorted(list(f"'{p}'" for p in i_ptmpro)) + ['/']))

In [38]:
with open('./model/rxn_compartments.txt', 'w') as loc_f:
	loc_f.write('/')
	for i in df_eqn.index:
		if pd.isna(df_eqn.get('compartment', {}).get(i, pd.NA)):
			continue
		loc_f.write(f"\n'{df_eqn.at[i, 'id']}'.'{df_eqn.at[i, 'compartment']}'")
	loc_f.write('\n/')

# Build GAMS RBA constraints (originally A03_build_GAMS_RBA_constraints.ipynb)

In [39]:
#### LOAD INPUTS AND PARAMETERS
copy_to_gams = True # if True, copy the model to the gams folder

# Stoichiometry
df_stoich = read_spreadsheet('./model/RBA_stoichiometry.tsv')
df_stoich.index = df_stoich.id.to_list()

In [40]:
### Write prosyn reaction
idx = [i for i in df_stoich.index if i[:7] == 'PROSYN-']
prosyn = ["'" + i + "'" for i in idx]
prosyn = ['/'] + prosyn + ['/']
with open('./model/RBA_rxns_prosyn.txt', 'w') as f:
	f.write('\n'.join(prosyn))
with open('./model/RBA_rxns_PTM.txt', 'w') as f:
	ptm_rxns = ["'" + i + "'" for i in df_stoich.index if i.startswith('PTM-')]
	f.write('\n'.join(['/'] + ptm_rxns + ['/']))
## write ribosyn set
idx = [i for i in df_stoich.index if i[:8] == 'RIBOSYN-']
ribosyn = ["'" + i + "'" for i in idx]
ribosyn = ['/'] + ribosyn + ['/']
with open('./model/RBA_rxns_ribosyn.txt', 'w') as f:
	f.write('\n'.join(ribosyn))

In [41]:
# Make fwd and reversible rxn list for metabolic network and all rxns (not needed but helpful for testing)
# Also, make flux coupling analysis file in case you use that for testing

# Using Patrick's FCA code
rxn_types = {'irrev': 0, 'reversible-fwd-half': 1, 'reversible-rev-half': 2, 'pseudoreaction': 3, 'exchange-fwd-half': 4, 'exchange-rev-half': 5}

rev_rxn_list = []; rev_list = []
fwd_rxn_list = []; fwd_list = []
fca_list = []

for i in df_stoich.id:
	tag,rxn_base_id,rxn_dir,enz_id = extract_details_from_rxnid(i)
	#print("'"+i+"' "+str(rxn_types['irrev']))
	#print(rxn_base_id)
	#print(i,'\t',tag,'\t',rxn_dir)
	if rxn_dir == 'FWD':
		fwd_list.append(i)
		if tag == 'RXN':
			fwd_rxn_list.append(i)
		if re.match(RBA_defaults.exchange_regex, rxn_base_id):
			fca_list.append("'"+i+"' "+str(rxn_types['exchange-fwd-half']))
		# Add as irreversible if no counterpart found in other direction
		elif i.replace("_FWD","_REV",1) not in df_stoich.id:
			fca_list.append("'"+i+"' "+str(rxn_types['irrev'])) 
		else:
			fca_list.append("'"+i+"' "+str(rxn_types['reversible-fwd-half']))
	elif rxn_dir == 'REV':
		rev_list.append(i)
		if tag == 'RXN':
			rev_rxn_list.append(i)
		if re.match(RBA_defaults.exchange_regex, rxn_base_id):
			fca_list.append("'"+i+"' "+str(rxn_types['exchange-rev-half']))
		# Add as irreversible if no counterpart found in other direction
		elif i.replace("_REV","_FWD",1) not in df_stoich.id:
			fca_list.append("'"+i+"' "+str(rxn_types['irrev'])) 
		else:
			fca_list.append("'"+i+"' "+str(rxn_types['reversible-rev-half']))
	else:
		fca_list.append("'"+i+"' "+str(rxn_types['irrev'])) 

fwd_rxn_list = ["'" + i + "'" for i in fwd_rxn_list if i != '/']
fwd_rxn_list = ['/'] + fwd_rxn_list + ['/']
with open('./model/RBA_rxns_rxnmetabolicnetworkFWD.txt', 'w') as f:
	f.write('\n'.join(fwd_rxn_list))
rev_rxn_list = ["'" + i + "'" for i in rev_rxn_list if i != '/']
rev_rxn_list = ['/'] + rev_rxn_list + ['/']
with open('./model/RBA_rxns_rxnmetabolicnetworkREV.txt', 'w') as f:
	f.write('\n'.join(rev_rxn_list))

fwd_list = ["'" + i + "'" for i in fwd_list if i != '/']
fwd_list = ['/'] + fwd_list + ['/']
with open('./model/RBA_rxns_FWD.txt', 'w') as f:
	f.write('\n'.join(fwd_list))
rev_list = ["'" + i + "'" for i in rev_list if i != '/']
rev_list = ['/'] + rev_list + ['/']
with open('./model/RBA_rxns_REV.txt', 'w') as f:
	f.write('\n'.join(rev_list))

fca_list = ['/'] + fca_list + ['/']
with open('./model/rxntype.txt', 'w') as f:
	f.write('\n'.join(fca_list))

In [42]:
# creating set pairing SM values w/ RBA ones, for use in converting flux constraints from SM format to RBA

# filter out rxns with no FBA_name
df_sm = df_stoich[df_stoich.FBA_name.notnull()]
# get SM rxn IDs from FBA_name column
sm_rxn_ids = df_sm.FBA_name.to_list()
# get RBA rxn IDs from index
rba_rxn_ids = df_sm.index.to_list()
rba_rxn_dirs = [extract_details_from_rxnid(i)[2] for i in rba_rxn_ids]
# convert dir to -1 if rev, 1 otherwise
rba_rxn_dirs = [-1 if i == 'REV' else 1 for i in rba_rxn_dirs]
# create list of strings pairing SM rxn IDs with RBA rxn IDs
sm_rba_rxn_pairs_dict = {rba_rxn_ids[i]:sm_rxn_ids[i] for i in range(0, len(sm_rxn_ids))}
# save as JSON file
with open('./model/SM_RBA_rxn_pairs.json', 'w') as f:
	json.dump(sm_rba_rxn_pairs_dict, f)
sm_rba_rxn_pairs = ["'"+sm_rxn_ids[i]+"'.'"+rba_rxn_ids[i]+"'"+' '+str(int(rba_rxn_dirs[i])) for i in range(0, len(sm_rxn_ids))]
# convert SM rxn bounds to RBA format; for each sm rxn, get the corresponding rba rxn and its direction
with open('./model/SM_rxn_bounds.txt', 'w') as f:
	delim = ' '
	sm_rxns_checked = []
	# find longest rxn ID for formatting purposes
	max_sm_rxn_id_len = max([len(i)+2 for i in sm_rxn_ids])+1
	max_lb_len = max([len(str(rxn.lower_bound)) for rxn in model.reactions if rxn.id in sm_rxn_ids])+1
	max_ub_len = max([len(str(rxn.upper_bound)) for rxn in model.reactions if rxn.id in sm_rxn_ids])
	# write headers to file, with ljust
	f.write(f"{'':>{max_sm_rxn_id_len}}{'v_sm_lo':>{max_lb_len}}{' v_sm_up':<{max_ub_len}}")
	for rba_rxn,sm_rxn in sm_rba_rxn_pairs_dict.items():
		# find bounds of sm rxn in model
		rxn = model.reactions.get_by_id(sm_rxn)
		ub = rxn.upper_bound
		lb = rxn.lower_bound

		if sm_rxn not in sm_rxns_checked:
			sm_rxns_checked.append(sm_rxn)
			name=f"'{sm_rxn}'"
			# make string with SM rxn ID, lb, and ub, using max lengths for formatting
			f.write(f"\n{name:>{max_sm_rxn_id_len}}{str(lb):>{max_lb_len}} {str(ub):<{max_ub_len}}")
# write to file
with open('./model/SM_RBA_rxn_pairs.txt', 'w') as f:
	f.write('\n'.join(['/'] + sorted(sm_rba_rxn_pairs) + ['/']))
# make file w/ all FBA IDs
with open('./model/SM_rxn_ids.txt', 'w') as f:
	f.write('\n'.join(['/'] + sorted(["'"+i+"'" for i in set(sm_rxn_ids)]) + ['/']))

In [43]:
# make file for all formula weights (some may be inaccurate due to R groups and other non-standard groups)
with open('./model/formula_weights.txt','w') as f:
	f.write('/\n')
	for met in model.metabolites:
		# write met name, formula, and formula weight
		try:
			formula = str(met.formula)
			mw = str(met.formula_weight)
		except:
			mw = 'unknown'
			formula = 'unknown'
		f.write(str(met.id) + '\t' + formula + '\t' + mw + '\n')
	f.write('/')

/Applications/anaconda3/envs/1pkrba/lib/python3.14/site-packages/cobra/core/metabolite.py:191: UserWarning: The element 'Tr' does not appear in the periodic table
  warn(f"The element {e} does not appear in the periodic table")
/Applications/anaconda3/envs/1pkrba/lib/python3.14/site-packages/cobra/core/metabolite.py:191: UserWarning: The element 'Ma' does not appear in the periodic table
  warn(f"The element {e} does not appear in the periodic table")
/Applications/anaconda3/envs/1pkrba/lib/python3.14/site-packages/cobra/core/metabolite.py:191: UserWarning: The element 'Fc' does not appear in the periodic table
  warn(f"The element {e} does not appear in the periodic table")
/Applications/anaconda3/envs/1pkrba/lib/python3.14/site-packages/cobra/core/metabolite.py:191: UserWarning: The element 'Ap' does not appear in the periodic table
  warn(f"The element {e} does not appear in the periodic table")
/Applications/anaconda3/envs/1pkrba/lib/python3.14/site-packages/cobra/core/metabolite.p

In [44]:
# for AA in df_aamap, find all mets in met_list that match its name (without the location)
with open('./model/aa.txt', 'w') as f, open('./model/tRNAs.txt', 'w') as f2:
	fis = [f,f2]
	for fi in fis: fi.write('/')
	for aa in df_aamap.index:
		aa_mets = [f"'{met}'" for met in met_list if met.rsplit('_',1)[0] == df_aamap.tRNA_in[aa].rsplit('_',1)[0]]
		f.write(f"\n'{aa}'")
		if aa_mets:
			f2.write(f"\n'{aa}'.({','.join(aa_mets)})")
	for fi in fis: fi.write('\n/')

In [45]:
# write combinations of gene_src and protein IDs in form 'gene_src'.('protein1','protein2',...)
with open('./model/proteins_and_locations.txt', 'w') as f, open('./model/proteins.txt', 'w') as f2:
	fis = [f,f2]
	for fi in fis: fi.write('/')
	for gene,prots in protein_functions.items():
		prots_str = ','.join([f"'PROSYN-{prot}'" for prot in prots])
		f.write(f"\n'{gene}'.({prots_str})")
		f2.write(f"\n'{gene}'")
	for fi in fis: fi.write('\n/')

In [46]:
# create version of model where all enzyme-catalyzed rxns (not including transport) are copied into all compartments

# print all enzyme-catalyzed reactions (not including transport) and their compartments

all_compartments = set()
all_subsystems = set()
for r in model.reactions:
	if r.subsystem.lower() not in ['transport','transport, extracellular','exchange','biomass','pseudoreaction']:
		# print(r.id, r.subsystem, r.compartments, r.metabolites)
		all_subsystems.add(r.subsystem)
		for met in r.metabolites:
			all_compartments.add(met.compartment)
# for met in model.metabolites:
# 	all_compartments.add(met.compartment)
all_compartments = sorted(list(all_compartments))
all_compartments.remove('e') # remove extracellular compartment
# all_compartments
all_subsystems

{'',
 'Acetyl-CoA synthesis',
 'Alanine, aspartate and glutamate metabolism',
 'Alternative carbon metabolism',
 'Amino sugar and nucleotide sugar metabolism',
 'Aminoacyl-tRNA biosynthesis',
 'Aminoacyl-tRNA biosynthesis (ec00970)',
 'Arachidonic acid metabolism',
 'Arginine and proline metabolism',
 'Biosynthesis of unsaturated fatty acids',
 'Biotin metabolism',
 'Butanoate metabolism',
 'C5-Branched dibasic acid metabolism',
 'Cell wall biosynthesis',
 'Citrate cycle (TCA cycle) (ec00020) | Cysteine and methionine metabolism (ec00270) | Pyruvate metabolism (ec00620) | Glyoxylate and dicarboxylate metabolism (ec00630) | Methane metabolism (ec00680) | Carbon fixation in photosynthetic organisms (ec00710) | Carbon fixation pathways in prokaryotes (ec00720) | Metabolic pathways (ec01100) | Biosynthesis of secondary metabolites (ec01110) | Microbial metabolism in diverse environments (ec01120) | Biosynthesis of antibiotics (ec01130)',
 'Citrate cycle (TCA cycle) (ec00020) | Glyoxylate a

In [47]:
if copy_to_gams: # automatically copies the model folder to the GAMS folder (removing the previous version), if you choose to
	import shutil
	# copy "model" folder to GAMS folder
	# Source folder path
	source_folder = './model'

	# Destination folder path
	destination_folder = '../GAMS/model'

	# Remove the existing destination folder
	if os.path.exists(destination_folder):
		shutil.rmtree(destination_folder)

	# Copy the folder
	shutil.copytree(source_folder, destination_folder)

In [48]:
# enzymatic_rxns = [] # all enzyme-catalyzed rxns, not including transport rxns
# for i in df_enz.index:
# 	rxn_id = df_enz.id[i]
# 	tag,rxn_base_id,rxn_dir,enz_id = extract_details_from_rxnid(rxn_id)
	
# 	if rxn_base_id[:3] == 'EX_' or tag != 'RXN':
# 		continue
# 	rxn = model.reactions.get_by_id(rxn_base_id)
# 	if rxn.subsystem.lower() not in ['transport','transport, extracellular','exchange','biomass','pseudoreaction'] and rxn.bounds != (0, 0):
# 		print(f"{rxn_base_id}")
# 		enzymatic_rxns.append(rxn_id)

In [49]:
# check if any enzymes likely have cofactors counted multiple times
for enz in set(df_enz['enz']):
	# find 1st row where the enz matches
	prot_stoich_str = df_enz[df_enz['enz'] == enz].iloc[0]['protein_stoich']
	if pd.isna(prot_stoich_str):
		continue
	# parse the protein_stoich string into a dictionary
	# print(enz, prot_stoich_str)
	prot_stoich = {item.split(':')[0]: float(item.split(':')[1]) for item in prot_stoich_str.split(',') if ':' in item and not pd.isna(item)}
	if not prot_stoich:
		continue
	rxns = set(df_enz[df_enz['enz'] == enz]['rxn_src'].to_list())
	# remove blocked reactions
	rxns = [rxn for rxn in rxns if model.reactions.get_by_id(rxn).bounds != (0, 0) or rxn in RBA_defaults.must_include_rxns]
	# find the trans_stoich_edits and ptm_stoich_edits for all proteins comprising it
	possible_cofactors = {}
	for pro in prot_stoich.keys():
		if pro in df_pro.index:
			met_dict = {}
			# Collect edits, filter out NaN, and convert to string
			edits = [df_pro.at[pro,'trans_stoich_edits'], df_pro.at[pro,'PTM_stoich_edits']] if 'trans_stoich_edits' in df_pro.columns else [df_pro.at[pro,'PTM_stoich_edits']]
			edits = [str(e) for e in edits if pd.notna(e)]
			if not edits:
				continue
			met_items = set(','.join(edits).split(','))
			met_dict = {i.split(':')[0]: float(i.split(':')[1]) for i in met_items if not pd.isna(i) and ':' in i}
			for k,v in met_dict.items():
				if v > 0: # i.e., if it's consumed
					if k in possible_cofactors:
						print(f"Warning: Enzyme {enz} has cofactor {k} counted multiple times, including subunits {possible_cofactors[k]} and {pro}.")
					possible_cofactors.setdefault(k, []).append(pro)
	if possible_cofactors:
		# print(enz,possible_cofactors)
		# find the reactions that have the same cofactors
		cofactors_in_rxns = set()
		for rxn in rxns:
			rxn_cofactors = set()
			# get the stoichiometry of the reaction
			rxn_stoich_dict = model.reactions.get_by_id(rxn).metabolites
			if rxn_stoich_dict is None:
				continue
			for met, coef in rxn_stoich_dict.items():
				# if coef < 0 and met.id in [i for i in possible_cofactors.keys()]:
				if coef < 0 and met.id.rsplit('_',1)[0] in [i.rsplit('_',1)[0] for i in possible_cofactors.keys()]:
					rxn_cofactors.add(met.id)
					cofactors_in_rxns.add(met.id)
		if cofactors_in_rxns:
			print(f"Enzyme {enz} has cofactors {cofactors_in_rxns} in its reactions.")
			# print the reactions that have the same cofactors
			for rxn in rxns:
				for cofactor in cofactors_in_rxns:
					if cofactor in [met.id for met in model.reactions.get_by_id(rxn).metabolites.keys()]:
						print(f"  - {rxn} has cofactor {cofactor}")
# # check for rxns that should be able to occur in a compartment but aren't modeled there
# # find all unique rxn stoichiometries in the model (not including exchange reactions)
# rxn_stoichs = {}
# for r in model.reactions:
# 	# get the stoichiometry of the reaction
# 	rxn_stoich_dict = r.metabolites
# 	if rxn_stoich_dict is None:
# 		continue
# 	# convert to a frozenset of tuples to make it hashable
# 	rxn_stoich = frozenset((met.id, coef) for met, coef in rxn_stoich_dict.items())
# 	rxn_stoichs.setdefault(rxn_stoich, []).append(r.id)
if warnings_summary:
	print('Warnings encountered during processing:')
	for warning in warnings_summary:
		print(warning)
	raise Warning ("Processing completed with warnings; please review the messages above, starting under 'Warnings encountered during processing:'.")

Enzyme JL09_g1388 has cofactors {'o2_c'} in its reactions.
  - TYMR_c has cofactor o2_c
  - MAOX_c has cofactor o2_c
  - AACTOOR_c has cofactor o2_c
  - PEAMNO_c has cofactor o2_c
  - NMPTRCOX_c has cofactor o2_c
  - PAO_c has cofactor o2_c
  - DAPO_c has cofactor o2_c
  - 4MEABUTNOX_c has cofactor o2_c
Enzyme JL09_g3871_c has cofactors {'o2_c'} in its reactions.
  - PAO_c has cofactor o2_c
  - DAPO_c has cofactor o2_c
  - MAOX_c has cofactor o2_c
